In [ ]:
# Setup cell. Canonical text: paste this file into the first code cell of every stage notebook.
STAGE = 6  # the only line that differs between notebooks

import contextlib
import json
import os
import shutil
import subprocess
import sys
import tarfile
import urllib.request
from pathlib import Path

ON_COLAB = "google.colab" in sys.modules
OBJECT = "my-object"  # folder name under real-to-sim-twin/ for your capture
RECORDING_ID = "real-to-sim-twin"
REFERENCE_URL = "https://artifacts.aegeanai.com/projects/real-to-sim-twin/reference"

RERUN = "rerun-sdk[notebook]==0.41.0"
PACKAGES = {
    1: ["pycolmap==4.2.1"],
    2: ["git+https://github.com/facebookresearch/sam2@2b90b9f", "huggingface_hub", "gsplat==1.5.3", "ninja",
        "git+https://github.com/rmbrualla/pycolmap@cc7ea4b"],
    3: ["gsplat==1.5.3", "ninja", "open3d==0.20.0", "git+https://github.com/rmbrualla/pycolmap@cc7ea4b"],
    4: ["diffusers==0.42.1", "transformers==5.21.0", "xatlas==0.0.11", "open3d==0.20.0", "pycolmap==4.2.1"],
    "4b": ["lpips==0.1.4", "pycolmap==4.2.1", "ninja"],
    5: ["trimesh==5.2.0", "coacd==1.0.14"],
    6: ["newton[sim,importers]==1.7.0", "usd-core==26.8", "trimesh==5.2.0", "lpips==0.1.4", "gsplat==1.5.3", "ninja",
        "pycolmap==4.2.1", "open3d==0.20.0"],
}
# Built against the installed torch, so pip must not isolate the build.
NO_BUILD_ISOLATION = {"4b": ["git+https://github.com/NVlabs/nvdiffrast@v0.4.0"],
                      6: ["git+https://github.com/NVlabs/nvdiffrast@v0.4.0"]}
# Reference objects with a course-made TRELLIS.2 archive, generated-<object>.tar.gz.
GENERATED_OBJECTS = ("boxed_beverage_001", "hammer_014")
# The TRELLIS.2 model revision those archives were generated with (model_revision in trellis2.json).
GENERATED_MODEL_REVISION = "af44b45f2e35a493886929c6d786e563ec68364d"


def _pip(*packages):
    subprocess.run([sys.executable, "-m", "pip", "install", "-q", *packages], check=True)


def _patch_pycolmap_fork():
    # The fork breaks on numpy 2: np.uint64(-1) raises OverflowError.
    import importlib
    import importlib.util
    importlib.invalidate_caches()  # the package was installed by pip in this process
    root = Path(importlib.util.find_spec("pycolmap").submodule_search_locations[0])
    for f in root.rglob("*.py"):
        text = f.read_text()
        if "np.uint64(-1)" in text:
            f.write_text(text.replace("np.uint64(-1)", "np.uint64(2**64-1)"))


if ON_COLAB:
    from google.colab import drive
    drive.mount("/content/drive")
    _pip(RERUN, *PACKAGES[STAGE])
    if STAGE in NO_BUILD_ISOLATION:
        _pip("--no-build-isolation", *NO_BUILD_ISOLATION[STAGE])
    if STAGE in (2, 3):  # stages 2 and 3 read stage1/sparse/0 with the pycolmap fork, never the official wheel
        _patch_pycolmap_fork()
    PROJECT = Path(f"/content/drive/MyDrive/real-to-sim-twin/{OBJECT}")
else:
    if "R2S_PROJECT" not in os.environ:
        raise RuntimeError("Off Colab, set the R2S_PROJECT environment variable to the project folder.")
    PROJECT = Path(os.environ["R2S_PROJECT"])
PROJECT.mkdir(parents=True, exist_ok=True)


@contextlib.contextmanager
def atomic_dir(path):
    """Yield path.tmp; on success it replaces path, on failure path is untouched."""
    path = Path(path)
    tmp, old = path.with_name(path.name + ".tmp"), path.with_name(path.name + ".old")
    shutil.rmtree(tmp, ignore_errors=True)
    if old.exists() and not path.exists():
        old.rename(path)  # a crash between the two renames left the only good copy in .old
    shutil.rmtree(old, ignore_errors=True)
    tmp.mkdir(parents=True)
    try:
        yield tmp
        if path.exists():
            path.rename(old)
        tmp.rename(path)
    except BaseException:
        shutil.rmtree(tmp, ignore_errors=True)
        if old.exists() and not path.exists():
            old.rename(path)
        raise
    shutil.rmtree(old, ignore_errors=True)


def prev_folder(stage):
    """The folder a stage reads: capture/ for stage 1, stage4/ for stage 4b, stage<N-1>/ otherwise.

    Stage 5 reads stage4/ too: stage 4b refines the maps but leaves the mesh to stage 4.
    """
    if stage == 1:
        return "capture"
    if stage == "4b":
        return "stage4"
    return f"stage{stage - 1}"


def on_reference(stage):
    """True if this stage runs on the reference object.

    The reference archives mark each stage folder with an empty .reference file, and a stage
    run on the reference marks the folder it writes. So the answer comes from the stage's own
    input folder (prev_folder(stage)), not from ground_truth/, which
    stays in the project after any fallback. For stage 1 a marked capture/ that also holds files
    of your own (own_capture_files) is not the reference.
    """
    if not (PROJECT / prev_folder(stage) / ".reference").is_file():
        return False
    return not (stage == 1 and own_capture_files(PROJECT / "capture"))


VIDEO_SUFFIXES = (".mp4", ".mov", ".m4v", ".avi", ".mkv", ".webm")


def own_capture_files(capture) -> list:
    """Files in capture/ that the reference capture never holds, so they must be yours.

    The reference capture has no board.json, and its only video is video.mp4, extracted at the same
    time as its .reference marker. So board.json, a video under another name (an iPhone's .mov, say)
    or a video changed more than a minute after the marker means that you added your own capture to
    a folder where a reference run left its files. board.json is the signal that does not depend on
    file times; your own capture always needs it for the metric scale.
    """
    capture = Path(capture)
    marker = capture / ".reference"
    since = marker.stat().st_mtime if marker.exists() else float("-inf")
    own = ["board.json"] if (capture / "board.json").exists() else []
    for p in sorted(capture.glob("*")):
        if p.suffix.lower() in VIDEO_SUFFIXES and (p.name != "video.mp4" or p.stat().st_mtime > since + 60):
            own.append(p.name)
    return own


def capture_video(capture):
    """The capture's video: video.mp4, or the one video file of another common phone format."""
    capture = Path(capture)
    if (capture / "video.mp4").exists():
        return capture / "video.mp4"
    videos = [p for p in sorted(capture.glob("*")) if p.suffix.lower() in VIDEO_SUFFIXES]
    if len(videos) != 1:
        raise FileNotFoundError(f"capture/ needs exactly one video (video.mp4, or one .mov, .m4v, .avi, .mkv or "
                                f".webm file); it holds {[p.name for p in videos] or 'none'}")
    return videos[0]


def _extract_reference(archive, names=None, dest=None):
    """Download a reference archive and move its top-level folders (or only `names`) into dest.

    dest defaults to PROJECT. A non-empty folder of yours that a reference folder replaces is
    kept as <name>.yours (an older .yours is replaced); ground_truth/ is replaced without a copy.
    Nothing moves until the archive is fully extracted.
    """
    dest = PROJECT if dest is None else Path(dest)
    base = os.environ.get("R2S_REFERENCE_URL", REFERENCE_URL).rstrip("/")
    tmp = dest / f".{archive}.part"
    req = urllib.request.Request(f"{base}/{archive}", headers={"User-Agent": "real-to-sim-twin"})
    with urllib.request.urlopen(req) as r, open(tmp, "wb") as f:
        shutil.copyfileobj(r, f)
    scratch = dest / ".extract"
    shutil.rmtree(scratch, ignore_errors=True)
    try:
        with tarfile.open(tmp) as tar:
            tar.extractall(scratch, filter="data")
        for item in scratch.iterdir():  # move into place only after a complete extraction
            if names is not None and item.name not in names:
                continue
            target = dest / item.name
            if target.is_dir() and any(target.iterdir()) and item.name != "ground_truth":
                yours = dest / f"{item.name}.yours"
                shutil.rmtree(yours, ignore_errors=True)
                target.rename(yours)
                print(f"your {item.name}/ is kept as {yours.name}/ and replaced by the reference {item.name}/.")
            elif target.is_dir():
                shutil.rmtree(target)
            elif target.exists():
                target.unlink()
            item.rename(target)
    finally:
        shutil.rmtree(scratch, ignore_errors=True)
        tmp.unlink(missing_ok=True)


def ensure_reference_capture(stage):
    """Stages 5 and 6 on the reference object use the reference capture/ too. True if it fetched it.

    When the stage's input folder is the reference object's (on_reference(stage), which follows
    the .reference markers from any earlier fallback) but capture/ is not (no capture/.reference),
    your measurements in capture/ belong to a different object than the mesh. The reference
    capture/ (physical.json, and the second-light files for stage 6) is then taken from the
    previous stage's archive, and yours is kept as capture.yours/.
    """
    if stage not in (5, 6) or not on_reference(stage) or (PROJECT / "capture" / ".reference").is_file():
        return False
    archive = f"{prev_folder(stage)}.tar.gz"
    print(f"{prev_folder(stage)}/ holds the reference object, so this stage uses the reference capture/ ({archive}).")
    _extract_reference(archive, names={"capture"})
    return True


def ensure_inputs(stage):
    """True if it fell back to the reference object, False if your inputs were present.

    The reference archive for a stage is cumulative: it holds every earlier stage folder that
    later stages read, plus ground_truth/, so a fallback gives one consistent object (see
    _extract_reference for what happens to folders of yours). In stages 5 and 6 it then makes
    sure that a reference object also gets the reference capture/ (ensure_reference_capture).
    """
    prev = prev_folder(stage)
    folder = PROJECT / prev
    fell_back = not (folder.is_dir() and any(folder.iterdir()))
    if fell_back:
        archive = "stage0.tar.gz" if stage == 1 else f"{prev}.tar.gz"
        print(f"{prev}/ is missing or empty: running on the reference object ({archive}).")
        _extract_reference(archive)
    ensure_reference_capture(stage)
    return fell_back


def ensure_generated(obj):
    """Fetch the course's TRELLIS.2 archive for a reference object and return its folder.

    generated-<obj>.tar.gz holds that object's stage folders, ground_truth/ and trellis2/. They
    go to generated/<obj>/, which is replaced whole on each download, so your own stage folders
    are never touched. A failed download leaves the previous generated/<obj>/ as it was. When
    generated/<obj>/trellis2/trellis2.json is already there with GENERATED_MODEL_REVISION, the
    folder is kept and nothing is downloaded.
    """
    if obj not in GENERATED_OBJECTS:
        raise ValueError(f"no generated archive for {obj!r}; the course ships {', '.join(GENERATED_OBJECTS)}")
    out = PROJECT / "generated" / obj
    try:
        revision = json.loads((out / "trellis2" / "trellis2.json").read_text()).get("model_revision")
    except (OSError, ValueError, AttributeError):
        revision = None
    if revision == GENERATED_MODEL_REVISION:
        print(f"generated/{obj}/ is already here (TRELLIS.2 revision {revision[:7]}), so it is not downloaded again.")
        return out
    with atomic_dir(out) as tmp:
        _extract_reference(f"generated-{obj}.tar.gz", dest=tmp)
    return out


import rerun as rr

RRD_PATH = PROJECT / "rrd" / f"stage{STAGE}.rrd"
RRD_PATH.parent.mkdir(parents=True, exist_ok=True)
rr.init("real-to-sim-twin", recording_id=RECORDING_ID)
rr.save(RRD_PATH)
rr.log("world", rr.ViewCoordinates.RIGHT_HAND_Z_UP, static=True)


def show_rerun():
    """Show everything logged so far in an inline viewer. Call it last in the stage.

    The viewer replaces the file sink, which leaves stage<N>.rrd complete; the viewer only gets
    data logged after it attaches, so the file is replayed into it. Later log calls reach the
    viewer only, not the file.
    """
    rr.notebook_show()
    rr.log_file_from_path(RRD_PATH)

In this section you package the twin as one asset that a simulator can load, then test it against the real object twice: once for how it looks and once for how it behaves. The asset is an OpenUSD layer (Pixar's Universal Scene Description, Tomorrow Open Source Technology License) that holds the textured mesh from stage 4, the convex pieces and mass properties from stage 5, and the friction coefficient from your tilt test. You drop it and slide it in the Newton simulator (Apache-2.0), which runs on NVIDIA Warp (Apache-2.0).

The appearance test renders the stage 3 splat [@Kerbl2023-rg] and the textured mesh from the frames that no stage trained on and scores each against the photo with PSNR and LPIPS (Zhang et al., "The Unreasonable Effectiveness of Deep Features as a Perceptual Metric", CVPR 2018; BSD-2-Clause code and AlexNet-based weights). When stage 4b ran, the base color of its refined maps is scored the same way beside the stage 4 bake, while the asset keeps the bake. The mesh is rasterized with nvdiffrast (Laine et al., "Modular Primitives for High-Performance Differentiable Rendering", SIGGRAPH Asia 2020), whose license is the NVIDIA Source Code License, for non-commercial use. The behavior test compares the measured mass, dimensions and slide angle with what the asset and the simulator give. Both results go into two scorecards, and the logs of every stage are merged into one recording. A last comparison scores a 3D model generated from a single photo against the measured twin, on the two reference objects the course ships.

Runtime budget on a Colab T4: about 4 minutes once the session is set up, about 2 minutes of it in the ramp sweep. The download of the two archives of the generated objects comes on top of that. The first run in a new session takes about 20 minutes, of which about 16 minutes is one-time setup.

In [ ]:
import json
import shutil
import time

import cv2
import numpy as np

T0 = time.time()
FALLBACK = ensure_inputs(STAGE)
CAPTURE, STAGE1, STAGE2, STAGE3, STAGE4, STAGE5 = (PROJECT / n for n in
                                                   ("capture", "stage1", "stage2", "stage3", "stage4", "stage5"))
REFERENCE = on_reference(STAGE)  # the reference stage5/ carries a .reference marker
physical = json.loads((CAPTURE / "physical.json").read_text())
physics = json.loads((STAGE5 / "physics.json").read_text())
scale = json.loads((STAGE1 / "scale.json").read_text())
WORK = PROJECT / "stage6.work"  # everything is built here; the check cell moves it into stage6/ in one step
shutil.rmtree(WORK, ignore_errors=True)
(WORK / "asset").mkdir(parents=True)
SCAN_OBJECT = json.loads((PROJECT / "ground_truth" / "scan.json").read_text())["object"] if REFERENCE else None
print("object:", f"reference ({SCAN_OBJECT})" if REFERENCE else OBJECT)
print(f"stage 5 exports the {physics['model'].replace('_', '-')} mass model: {physics['mass_kg']} kg, center of mass "
      f"{np.round(np.asarray(physics['com_m']) * 1e3, 2)} mm, mu = {physics['mu_static']:.3f}")
print(f"stage 1 scale source: {scale['source']}")
if physical.get("assumed"):
    print("physical.json holds assumed values, not measurements, for this object")

## The asset as a USD layer

A USD layer describes a scene as a tree of *prims*, each with a path such as `/object/visual`, a type, and typed attributes. Behavior is added by applying *schemas* to a prim. The tree you write has one root, `/object`, which is the rigid body, with three groups under it: the visual mesh, the collision pieces, and the materials.

The root carries `RigidBodyAPI`, which tells a simulator to integrate this prim and everything under it as one body, and `MassAPI`, which gives the mass $m$, the center of mass $c$ and the inertia. USD does not store the full inertia tensor. It stores the tensor in its own principal frame: the three principal moments, which are the eigenvalues of $I$, and the rotation of that frame, as a unit quaternion. The two descriptions are related by the eigendecomposition

$$
I = R \,\mathrm{diag}(I_1, I_2, I_3)\, R^\top ,
$$

where the columns of $R$ are the eigenvectors of $I$. An eigenvector is defined only up to its sign, so the solver may return a matrix with determinant $-1$, a reflection. A quaternion can only describe a rotation, so you flip one column to make $\det R = +1$ before converting. A simulator rebuilds $I$ from the moments and the quaternion. If the order of moments and axes is mixed up, the body still has the right total mass but turns about the wrong axes.

Each convex piece becomes its own mesh prim with `CollisionAPI`, and `MeshCollisionAPI` with the approximation `convexHull` tells the simulator to collide the hull of those points, which is the piece itself. The pieces are marked as guides, so a renderer does not draw them. They are bound to a physics material whose static and dynamic friction are your $\mu$ and whose restitution is 0. The visual mesh is bound to a preview surface, a small physically based shading model whose inputs are the base color, roughness and metallic maps of stage 4, read through the mesh's texture coordinates. USD's texture coordinates put $v = 0$ at the bottom of the image, so a coordinate $v$ measured from the top becomes $1 - v$.

A metallic value of 1 tells the renderer that the surface reflects like a metal: no diffuse color, and specular highlights tinted by the base color. Stage 4 found that the metallic estimate is unreliable on non-metal objects. On the reference carton, which is printed paperboard, its mean is about 0.4. The export therefore keeps the baked metallic map only when `physical.json` names materials, which is your statement that part of the object is metal. Otherwise the metallic input is set to 0. A metal object without named materials then renders as non-metal; naming its materials restores the map.

The stage also records its units. `metersPerUnit = 1` and `kilogramsPerUnit = 1` say that the numbers are meters and kilograms, and `upAxis = Z` says which way is up. The world frame of stage 1 is metric with Z up, so nothing is converted. A simulator that assumed centimeters or a Y-up world would otherwise scale the object by 100 or lay it on its side.

After writing, the layer is checked by OpenUSD's validators, which test the stage metadata, the material bindings, the shader inputs and the physics schemas, and then read back to compare every physical value with stage 5.


In [ ]:
import json
import shutil
from pathlib import Path

import numpy as np
from pxr import Gf, Sdf, Usd, UsdGeom, UsdPhysics, UsdShade, Vt


def load_mesh_uv(path: Path) -> dict:
    """Read stage4/mesh_uv.obj: "vertices" (V x 3, meters), "faces" (F x 3), "normals" (V x 3) and
    "uvs" (V x 2, origin at the top-left of the texture, v down; the OBJ stores v up)."""
    v, vt, vn, f = [], [], [], []
    for line in Path(path).read_text().splitlines():
        if line.startswith("v "):
            v.append(line.split()[1:4])
        elif line.startswith("vt "):
            vt.append(line.split()[1:3])
        elif line.startswith("vn "):
            vn.append(line.split()[1:4])
        elif line.startswith("f "):
            f.append([int(w.split("/")[0]) - 1 for w in line.split()[1:4]])
    uvs = np.asarray(vt, dtype=float)
    if len(uvs) != len(v):  # stage 4 writes f a/a/a: vertex i has texture coordinate i
        raise ValueError(f"{path} has {len(v)} vertices and {len(uvs)} texture coordinates; stage 6 reads the "
                         "stage 4 format, one texture coordinate per vertex")
    return {"vertices": np.asarray(v, dtype=float), "faces": np.asarray(f, dtype=np.int64),
            "normals": np.asarray(vn, dtype=float) if len(vn) == len(v) else None,
            "uvs": np.c_[uvs[:, 0], 1 - uvs[:, 1]]}


def load_pieces(folder: Path) -> list:
    """The stage 5 convex pieces, as (vertices, faces) pairs in meters."""
    pieces = []
    for path in sorted(Path(folder).glob("piece_*.obj")):
        v, f = [], []
        for line in path.read_text().splitlines():
            if line.startswith("v "):
                v.append(line.split()[1:4])
            elif line.startswith("f "):
                f.append([int(w.split("/")[0]) - 1 for w in line.split()[1:4]])
        pieces.append((np.asarray(v, dtype=float), np.asarray(f, dtype=np.int64)))
    return pieces


def quat_from_matrix(R) -> Gf.Quatf:
    """Unit quaternion (real part first) of a proper rotation matrix that acts on column vectors."""
    R = np.asarray(R, dtype=float)
    t = np.trace(R)
    if t > max(R[0, 0], R[1, 1], R[2, 2]):
        w = np.sqrt(1 + t) / 2
        x, y, z = (R[2, 1] - R[1, 2]) / (4 * w), (R[0, 2] - R[2, 0]) / (4 * w), (R[1, 0] - R[0, 1]) / (4 * w)
    elif R[0, 0] >= R[1, 1] and R[0, 0] >= R[2, 2]:
        x = np.sqrt(1 + R[0, 0] - R[1, 1] - R[2, 2]) / 2
        w, y, z = (R[2, 1] - R[1, 2]) / (4 * x), (R[0, 1] + R[1, 0]) / (4 * x), (R[0, 2] + R[2, 0]) / (4 * x)
    elif R[1, 1] >= R[2, 2]:
        y = np.sqrt(1 - R[0, 0] + R[1, 1] - R[2, 2]) / 2
        w, x, z = (R[0, 2] - R[2, 0]) / (4 * y), (R[0, 1] + R[1, 0]) / (4 * y), (R[1, 2] + R[2, 1]) / (4 * y)
    else:
        z = np.sqrt(1 - R[0, 0] - R[1, 1] + R[2, 2]) / 2
        w, x, y = (R[1, 0] - R[0, 1]) / (4 * z), (R[0, 2] + R[2, 0]) / (4 * z), (R[1, 2] + R[2, 1]) / (4 * z)
    return Gf.Quatf(float(w), float(x), float(y), float(z))


def matrix_from_quat(q) -> np.ndarray:
    """Rotation matrix of a Gf quaternion (the inverse of quat_from_matrix)."""
    w, (x, y, z) = q.GetReal(), q.GetImaginary()
    return np.array([[1 - 2 * (y * y + z * z), 2 * (x * y - z * w), 2 * (x * z + y * w)],
                     [2 * (x * y + z * w), 1 - 2 * (x * x + z * z), 2 * (y * z - x * w)],
                     [2 * (x * z - y * w), 2 * (y * z + x * w), 1 - 2 * (x * x + y * y)]])


def define_mesh(stage, path: str, vertices, faces, st=None) -> UsdGeom.Mesh:
    """A triangle mesh prim with its extent, and texture coordinates st (V x 2, v up) if given."""
    mesh = UsdGeom.Mesh.Define(stage, path)
    v = np.asarray(vertices, dtype=np.float32)
    mesh.CreatePointsAttr(Vt.Vec3fArray.FromNumpy(v))
    mesh.CreateFaceVertexCountsAttr(Vt.IntArray.FromNumpy(np.full(len(faces), 3, dtype=np.int32)))
    mesh.CreateFaceVertexIndicesAttr(Vt.IntArray.FromNumpy(np.asarray(faces, dtype=np.int32).ravel()))
    mesh.CreateSubdivisionSchemeAttr(UsdGeom.Tokens.none)
    mesh.CreateExtentAttr([Gf.Vec3f(*map(float, v.min(0))), Gf.Vec3f(*map(float, v.max(0)))])
    if st is not None:
        primvar = UsdGeom.PrimvarsAPI(mesh).CreatePrimvar("st", Sdf.ValueTypeNames.TexCoord2fArray,
                                                          UsdGeom.Tokens.vertex)
        primvar.Set(Vt.Vec2fArray.FromNumpy(np.asarray(st, dtype=np.float32)))
    return mesh


def preview_material(stage, path: str, folder: Path, textures: dict) -> UsdShade.Material:
    """A UsdPreviewSurface material. textures maps "base_color", "roughness" and "metallic" to a
    PNG path (copied next to the USD file and read through the mesh's st coordinates) or to a constant."""
    material = UsdShade.Material.Define(stage, path)
    surface = UsdShade.Shader.Define(stage, f"{path}/surface")
    surface.CreateIdAttr("UsdPreviewSurface")
    material.CreateSurfaceOutput().ConnectToSource(surface.ConnectableAPI(), "surface")
    reader = UsdShade.Shader.Define(stage, f"{path}/st_reader")
    reader.CreateIdAttr("UsdPrimvarReader_float2")
    st_name = material.CreateInput("frame:stPrimvarName", Sdf.ValueTypeNames.String)
    st_name.Set("st")
    reader.CreateInput("varname", Sdf.ValueTypeNames.String).ConnectToSource(st_name)
    reader.CreateOutput("result", Sdf.ValueTypeNames.Float2)
    slots = {"base_color": ("diffuseColor", Sdf.ValueTypeNames.Color3f, "rgb", "sRGB"),
             "roughness": ("roughness", Sdf.ValueTypeNames.Float, "r", "raw"),
             "metallic": ("metallic", Sdf.ValueTypeNames.Float, "r", "raw")}
    for name, (slot, kind, channel, color_space) in slots.items():
        source = textures[name]
        if isinstance(source, (int, float)):
            surface.CreateInput(slot, kind).Set(Gf.Vec3f(float(source)) if channel == "rgb" else float(source))
            continue
        target = Path(folder) / Path(source).name
        if Path(source).resolve() != target.resolve():
            shutil.copyfile(source, target)
        texture = UsdShade.Shader.Define(stage, f"{path}/{name}")
        texture.CreateIdAttr("UsdUVTexture")
        texture.CreateInput("file", Sdf.ValueTypeNames.Asset).Set(f"./{target.name}")
        texture.CreateInput("sourceColorSpace", Sdf.ValueTypeNames.Token).Set(color_space)
        texture.CreateInput("st", Sdf.ValueTypeNames.Float2).ConnectToSource(reader.ConnectableAPI(), "result")
        out = texture.CreateOutput(channel, Sdf.ValueTypeNames.Float3 if channel == "rgb" else Sdf.ValueTypeNames.Float)
        surface.CreateInput(slot, kind).ConnectToSource(out)
    return material


def validate_usd(path: Path) -> list[str]:
    """Errors from OpenUSD's validators (UsdValidation, which replaced UsdUtils.ComplianceChecker),
    leaving out the checks that apply only to .usdz packages."""
    from pxr import UsdValidation

    registry = UsdValidation.ValidationRegistry()
    validators = [v for v in registry.GetOrLoadAllValidators()
                  if "UsdzValidators" not in v.GetMetadata().GetKeywords()]
    errors = UsdValidation.ValidationContext(validators).Validate(Usd.Stage.Open(str(path)))
    return [f"{e.GetName()}: {e.GetMessage()}" for e in errors
            if e.GetType() in (UsdValidation.ValidationErrorType.Error, UsdValidation.ValidationErrorType.Warn)]


def read_usd(path: Path) -> dict:
    """What a simulator will read from the asset: units, up axis, mass properties, colliders, friction."""
    stage = Usd.Stage.Open(str(path))
    root = stage.GetDefaultPrim()
    mass = UsdPhysics.MassAPI(root)
    colliders, friction = [], set()
    for prim in stage.Traverse():
        if prim.HasAPI(UsdPhysics.CollisionAPI):
            colliders.append({"path": str(prim.GetPath()),
                              "approximation": UsdPhysics.MeshCollisionAPI(prim).GetApproximationAttr().Get(),
                              "points": np.asarray(UsdGeom.Mesh(prim).GetPointsAttr().Get(), dtype=float)})
            bound = UsdShade.MaterialBindingAPI(prim).ComputeBoundMaterial("physics")[0]
            if bound:
                api = UsdPhysics.MaterialAPI(bound.GetPrim())
                friction.add((api.GetStaticFrictionAttr().Get(), api.GetDynamicFrictionAttr().Get()))
    R = matrix_from_quat(mass.GetPrincipalAxesAttr().Get())
    diag = np.asarray(mass.GetDiagonalInertiaAttr().Get(), dtype=float)
    return {"meters_per_unit": UsdGeom.GetStageMetersPerUnit(stage), "up_axis": UsdGeom.GetStageUpAxis(stage),
            "rigid_body": root.HasAPI(UsdPhysics.RigidBodyAPI), "mass_kg": mass.GetMassAttr().Get(),
            "com_m": np.asarray(mass.GetCenterOfMassAttr().Get(), dtype=float), "inertia_kgm2": R @ np.diag(diag) @ R.T,
            "colliders": colliders, "friction": sorted(friction)}

In [ ]:
import numpy as np
from pathlib import Path


def write_usd(path, mesh_uv: dict, textures: dict, pieces: list, physics: dict) -> None:
    """Write the asset as one USD layer at path, Z up, in meters and kilograms.

    The default prim /object is an Xform with UsdPhysics.RigidBodyAPI and MassAPI: mass, center of
    mass, and the inertia tensor of physics["inertia_kgm2"] as its eigenvalues (diagonalInertia)
    and the rotation whose columns are its eigenvectors (principalAxes, quat_from_matrix; make the
    rotation proper first). Under it:
      /object/visual: the textured mesh (define_mesh, st = (u, 1 - v)) bound to a preview_material
        at /object/Looks/pbr with the textures dict;
      /object/collision/piece_NN: one mesh per convex piece, purpose "guide", with CollisionAPI,
        MeshCollisionAPI approximation convexHull, and the physics material bound for "physics";
      /object/Looks/physics: a UsdShade.Material with UsdPhysics.MaterialAPI, static and dynamic
        friction physics["mu_static"], restitution 0.
    Set the stage's up axis, metersPerUnit and kilogramsPerUnit, save the root layer.
    """
    raise NotImplementedError

In [ ]:
# Metallic: keep the baked map only when physical.json names materials; otherwise the object is
# treated as non-metal and the shader's metallic input is 0.
metallic_source = "baked" if "materials" in physical else 0.0
textures = {"base_color": STAGE4 / "base_color.png", "roughness": STAGE4 / "roughness.png",
            "metallic": STAGE4 / "metallic.png" if metallic_source == "baked" else 0.0}
baked_metallic = cv2.imread(str(STAGE4 / "metallic.png"), cv2.IMREAD_GRAYSCALE) / 255.0
print(f"metallic: {'baked map kept (physical.json names materials)' if metallic_source == 'baked' else 'set to 0'}"
      f"; the baked map's mean is {baked_metallic.mean():.2f}"
      + ("" if metallic_source == "baked" else ", but physical.json names no materials, so the object is non-metal"))
mesh_uv = load_mesh_uv(STAGE4 / "mesh_uv.obj")
pieces = load_pieces(STAGE5 / "collision")
USD = WORK / "asset" / "object.usda"
t = time.time()
write_usd(USD, mesh_uv, textures, pieces, physics)
asset_mb = sum(f.stat().st_size for f in USD.parent.iterdir()) / 1e6
print(f"wrote {USD.name} with {len(mesh_uv['faces']):,} visual triangles and {len(pieces)} convex colliders in "
      f"{time.time() - t:.1f} s; the asset folder holds {asset_mb:.1f} MB")
usd_errors = validate_usd(USD)
print("OpenUSD validators:", "no errors or warnings" if not usd_errors else "; ".join(usd_errors))
asset = read_usd(USD)
print(f"read back: {asset['meters_per_unit']} m per unit, up axis {asset['up_axis']}, mass {asset['mass_kg']:.3f} kg, "
      f"{len(asset['colliders'])} colliders ({', '.join(sorted({c['approximation'] for c in asset['colliders']}))}), "
      f"friction (static, dynamic) {[tuple(round(x, 3) for x in f) for f in asset['friction']]}")

## Drop and tilt in the simulator

A rigid-body simulator advances positions and velocities in small time steps. At each step it finds the points where two collision shapes touch and corrects the bodies so that they do not interpenetrate. Newton's XPBD solver (extended position-based dynamics) does this on positions: it moves the bodies apart along the contact normal by the penetration depth, and it applies friction as a positional correction along the surface, limited by $\mu$ times the normal correction. This is the discrete form of Coulomb's law. The tangential force can hold the body as long as it stays below $\mu N$, and the body slides once it would have to exceed it.

The drop test places the object in its exported pose a few centimeters above a ground plane and lets it fall. It comes to rest if its center of mass moves less than half a millimeter and it turns less than 0.005 radians over the last tenth of a second. An object that keeps rocking or drifting has colliders that do not meet the ground in a stable face, or an inertia tensor that does not match its mass.

The tilt test repeats your ramp measurement. The object is laid in its lowest stable pose: of the faces of the colliders' convex hull, the stable ones are those that contain the projection of the center of mass, and the lowest center of mass among them is how you would lay the object down. On a ramp at angle $\theta$, stage 5 showed that it slides once $\tan\theta > \mu$. The simulation steps the ramp from 0 to 60 degrees in 1 degree increments. At each angle the object settles for 0.2 seconds and then has one second to move; it slides if its center of mass moves more than 5 mm along the ramp. The first such angle is the simulated slide angle, and you compare it with your real one. There is no pass threshold. A position-based solver also creeps a little below the slip angle, because each step corrects a small tangential drift only after it has happened. The 5 mm criterion separates that creep from sliding, and the cell prints the creep at the last angle that holds.

The ramp's own friction is set to the same $\mu$. A contact between two shapes uses the average of their coefficients, so the pair keeps your measured value.


In [ ]:
import contextlib
import io

import numpy as np
from pathlib import Path


def _newton():
    import warp as wp

    wp.config.quiet = True  # before the first kernel load, or Warp prints its device table
    import newton

    return newton, wp


def _wp_quat(wp, R):
    q = quat_from_matrix(R)
    x, y, z = q.GetImaginary()
    return wp.quat(x, y, z, q.GetReal())


def rot_y(deg: float) -> np.ndarray:
    """Rotation about +y: a ramp tilted by deg, which goes downhill toward +x."""
    a = np.radians(deg)
    return np.array([[np.cos(a), 0, np.sin(a)], [0, 1, 0], [-np.sin(a), 0, np.cos(a)]])


def resting_pose(usd_path) -> np.ndarray:
    """The 4 x 4 transform that lays the object on z = 0 in its lowest stable pose.

    A pose is stable when the center of mass projects inside the face of the colliders' convex hull
    that touches the plane; of those, the one with the lowest center of mass is how you would lay the
    object down for a tilt test. The transform puts the center of mass above the origin.
    """
    import trimesh

    asset = read_usd(usd_path)
    points = np.concatenate([c["points"] for c in asset["colliders"]])
    hull = trimesh.convex.convex_hull(points)
    poses, _ = trimesh.poses.compute_stable_poses(hull, center_mass=asset["com_m"], sigma=0.0, n_samples=1)
    pose = min(poses, key=lambda T: (T[:3, :3] @ asset["com_m"] + T[:3, 3])[2])
    R = pose[:3, :3]
    out = np.eye(4)
    out[:3, :3] = R
    c = R @ asset["com_m"]
    out[:3, 3] = [-c[0], -c[1], -(points @ R.T)[:, 2].min()]
    return out


def simulate(builder, seconds: float, dt: float = 1 / 600, iterations: int = 10) -> dict:
    """Step the scene with Newton's XPBD solver; returns the body's center of mass over time (T x 3,
    world), its orientation (T x 3 x 3) and the time step. Gravity is set after the model is built:
    add_usd rotates a layer's own PhysicsScene gravity with the import transform. The asset has no
    PhysicsScene, so this only guards layers that bring one."""
    newton, wp = _newton()
    model = builder.finalize()
    model.set_gravity((0.0, 0.0, -9.81))
    solver = newton.solvers.SolverXPBD(model, iterations=iterations)
    state, next_state, control = model.state(), model.state(), model.control()
    pipeline = newton.CollisionPipeline(model)
    contacts = pipeline.contacts()
    com_local = model.body_com.numpy()[0].astype(float)
    centers, rotations = [], []
    steps = int(round(seconds / dt))
    for i in range(steps + 1):
        q = state.body_q.numpy()[0].astype(float)
        x, y, z, w = q[3:]
        R = np.array([[1 - 2 * (y * y + z * z), 2 * (x * y - z * w), 2 * (x * z + y * w)],
                      [2 * (x * y + z * w), 1 - 2 * (x * x + z * z), 2 * (y * z - x * w)],
                      [2 * (x * z - y * w), 2 * (y * z + x * w), 1 - 2 * (x * x + y * y)]])
        centers.append(q[:3] + R @ com_local)
        rotations.append(R)
        if i == steps:
            break
        pipeline.collide(state, contacts)
        state.clear_forces()
        solver.step(state, next_state, control, contacts, dt)
        state, next_state = next_state, state
    return {"com": np.array(centers), "rotation": np.array(rotations), "dt": dt,
            "mass_kg": float(model.body_mass.numpy()[0])}


def ramp_trial(usd_path, angle_deg: float, mu: float, rest=None, seconds: float = 1.0, settle: float = 0.2) -> dict:
    """Lay the object in its resting pose on a ramp tilted by angle_deg and simulate settle + seconds.

    The ramp's friction is mu; Newton averages the two shapes' coefficients, so with the asset's own
    mu the contact uses mu. Returns "com_ramp", the center of mass in the ramp's frame over time
    (x downhill, z along the ramp's normal), "settle_index", the sample where the measured second
    starts, and the simulation record.
    """
    newton, wp = _newton()
    rest = resting_pose(usd_path) if rest is None else rest
    Rr = rot_y(angle_deg)
    builder = newton.ModelBuilder()
    builder.add_shape_box(-1, xform=wp.transform(wp.vec3(*(Rr @ [0.0, 0.0, -0.05])), _wp_quat(wp, Rr)),
                          hx=1.0, hy=1.0, hz=0.05, cfg=newton.ModelBuilder.ShapeConfig(mu=mu))
    position = Rr @ (rest[:3, 3] + [0.0, 0.0, 0.0005])  # half a millimeter above the ramp
    with contextlib.redirect_stdout(io.StringIO()):
        builder.add_usd(str(usd_path), xform=wp.transform(wp.vec3(*position), _wp_quat(wp, Rr @ rest[:3, :3])),
                        floating=True)
    run = simulate(builder, settle + seconds)
    return {"com_ramp": run["com"] @ Rr, "settle_index": int(round(settle / run["dt"])), "run": run,
            "ramp_rotation": Rr}


def drop_test(usd_path, mu: float, height: float = 0.05, seconds: float = 1.5) -> dict:
    """Drop the object in its exported pose from height (lowest collider point above a ground plane)
    and report whether it comes to rest: over the last 0.1 s its center of mass stays within 0.5 mm
    and its orientation within 0.005 rad of where they end (5 mm/s and 0.05 rad/s)."""
    newton, wp = _newton()
    lowest = min(c["points"][:, 2].min() for c in read_usd(usd_path)["colliders"])
    builder = newton.ModelBuilder()
    builder.add_ground_plane(cfg=newton.ModelBuilder.ShapeConfig(mu=mu))
    with contextlib.redirect_stdout(io.StringIO()):
        builder.add_usd(str(usd_path), xform=wp.transform(wp.vec3(0.0, 0.0, height - lowest), wp.quat_identity()),
                        floating=True)
    run = simulate(builder, seconds)
    tail = int(round(0.1 / run["dt"]))
    # largest excursion from the final pose over the window, not per-step differences: a resting
    # contact still jitters by about g dt^2 per step, and a rocking body returns to where it was
    speed = np.linalg.norm(run["com"][-1 - tail:] - run["com"][-1], axis=1).max() / 0.1
    dR = np.einsum("tij,kj->tik", run["rotation"][-1 - tail:], run["rotation"][-1])
    spin = np.arccos(np.clip((np.trace(dR, axis1=1, axis2=2) - 1) / 2, -1, 1)).max() / 0.1
    up = run["rotation"][-1][:, 2]
    return {"rest": bool(speed < 0.005 and spin < 0.05), "final_speed_m_s": float(speed), "final_spin_rad_s": float(spin),
            "tilt_deg": float(np.degrees(np.arccos(np.clip(up[2], -1, 1)))),
            "drop_m": height, "com_z_m": float(run["com"][-1][2]), "run": run}

In [ ]:
import math

import numpy as np


def slide_angle_sim(usd_path, mu: float, start_deg: int = 0, stop_deg: int = 60, step_deg: int = 1) -> float:
    """First ramp angle (degrees) at which the simulated object slides, or nan if it never does.

    Lay the object in its resting pose once (rest = resting_pose(usd_path)), then for each angle from
    start_deg to stop_deg in steps of step_deg run trial = ramp_trial(usd_path, angle, mu, rest).
    The object slides when its center of mass moves more than 5 mm along the ramp's surface (the x
    and y columns of trial["com_ramp"]) in the one second after trial["settle_index"].
    """
    raise NotImplementedError

In [ ]:
t = time.time()
drop = drop_test(USD, physics["mu_static"])
print(f"drop from {drop['drop_m'] * 100:.0f} cm onto the ground: {'comes to rest' if drop['rest'] else 'does NOT come to rest'} "
      f"(speed {drop['final_speed_m_s'] * 1e3:.2f} mm/s, spin {drop['final_spin_rad_s']:.3f} rad/s over the last 0.1 s; "
      f"up axis tilted {drop['tilt_deg']:.1f} degrees; simulated mass {drop['run']['mass_kg']:.3f} kg)")
t = time.time()
sim_angle = slide_angle_sim(USD, physics["mu_static"])
sweep_s = time.time() - t
tilt = np.asarray(physical["tilt_deg"], dtype=float)
print(f"ramp sweep in {sweep_s:.0f} s: the simulated object first slides at {sim_angle:.0f} degrees")
print(f"real slide angle: {tilt.mean():.1f} degrees (mean of {len(tilt)} readings, spread {tilt.std(ddof=1):.1f}); "
      f"atan(mu) = {np.degrees(np.arctan(physics['mu_static'])):.1f} degrees")
if physical.get("assumed"):
    print("the real readings are assumed, not measured: physical.json says so")
# the last angle that holds and the first that slides, kept for the viewer
hold = ramp_trial(USD, max(sim_angle - 1, 0), physics["mu_static"]) if np.isfinite(sim_angle) else None
slide = ramp_trial(USD, sim_angle, physics["mu_static"]) if np.isfinite(sim_angle) else None
if hold is not None:
    creep = hold["com_ramp"][hold["settle_index"]:, :2]
    print(f"at {max(sim_angle - 1, 0):.0f} degrees the object creeps {np.linalg.norm(creep[-1] - creep[0]) * 1e3:.2f} mm in 1 s")

## Appearance on held-out views

Every tenth frame of stage 1 was held out: stages 2 to 5 used it for nothing but evaluation. A model scored on the views it was fitted to measures how well it memorized them. A splat can reach a high score on its training views with floaters and view-dependent color that look wrong from anywhere else. The held-out frames measure what you care about, which is how the twin looks from a new viewpoint.

The first score is the peak signal-to-noise ratio over the object's pixels $M$ (its stage 2 mask), with images in $[0, 1]$:

$$
\mathrm{PSNR} = -10 \log_{10} \frac{1}{3|M|} \sum_{p \in M} \lVert \hat{x}_p - x_p \rVert^2 \quad \text{dB}.
$$

PSNR compares pixels one by one. It punishes a one-pixel shift of a sharp edge as much as a visible change of color, and it prefers a blurred image to a sharp one that is slightly misaligned, because blur halves the error on both sides of an edge. LPIPS compares images the way a trained network sees them. Both images pass through the layers of a network trained for image classification (AlexNet). At each layer $l$ the activations are normalized along the channel axis and their difference is weighted per channel by $w_l$, a set of weights fitted to human judgments of which of two distorted images looks closer to a reference:

$$
\mathrm{LPIPS}(\hat{x}, x) = \sum_l \frac{1}{H_l W_l} \sum_{h, w} \left\lVert w_l \odot \left( \hat{y}^{\,l}_{hw} - y^{l}_{hw} \right) \right\rVert_2^2 .
$$

Zero means identical. Values around 0.1 are small differences, and values above 0.4 are images a person would call clearly different. Here both images are cut to the object's box, with everything outside its mask set to white, so that the background does not enter the score.

Both scores are computed for the 3D splat and for the textured mesh. The splat stores the photographed radiance, lighting included, so it can match a photo closely. The mesh carries the base color of stage 4, an albedo with the lighting removed, so it renders flatter and usually brighter than a photo. A third row multiplies the mesh render by the single gain that best fits each photo in linear intensity, which removes the overall exposure and keeps everything else. The same splat score on a sample of training frames gives the gap between seen and unseen views.

Stage 4b fitted new maps and a light to the training photos. When `stage4b/` holds those maps, two more rows score their base color exactly as the bake is scored: rendered with no light, and again after one gain per photo. Neither the bake rows nor these rows carry any light, so the comparison is between the two albedos alone. Although the refined maps were fitted to the training frames, on the reference object their training and held-out rows agree to within 0.05 dB, and their LPIPS is slightly worse on the training frames. Compare the two columns on your own object.

A last row is the refined maps rendered under the light that stage 4b learned. It is copied from `stage4b/refine.json` and not measured again here. Stage 4b scored it on these held-out frames with the same score functions but its own renderer, which differs from this one in how it rasterizes and samples the texture. The same maps can score differently in the two stages, by up to about 0.6 dB on the reference object, and the cause of that difference has not been isolated. So do not read what the light adds by setting the relit row against a row of this table. The relit line also prints stage 4b's gain-matched score of the same maps, from the same renderer, and the difference between those two numbers is what the learned light adds.

Stage 4b recorded a hash of the `stage4/mesh_uv.obj` it was fitted to. If you ran stage 4 again afterward, the refined maps belong to an older UV layout and would be painted onto the wrong triangles, so this section leaves their rows out and asks you to rerun stage 4b. A project without `stage4b/` leaves them out too. In every case the exported asset carries the stage 4 bake.


In [ ]:
import contextlib
import io

import cv2
import numpy as np
from pathlib import Path


def opencv_distortion(model: str, params) -> np.ndarray:
    """OpenCV's (k1, k2, p1, p2) for a COLMAP camera model and its parameter list."""
    params = list(params)
    if model in ("PINHOLE", "SIMPLE_PINHOLE"):
        return np.zeros(4)
    if model == "SIMPLE_RADIAL":
        return np.array([params[3], 0.0, 0.0, 0.0])
    if model == "RADIAL":
        return np.array([params[3], params[4], 0.0, 0.0])
    if model == "OPENCV":
        return np.array(params[4:8], dtype=float)
    raise ValueError(f"camera model {model} is not supported here")


def load_views(stage1: Path, frames, masks: dict, data_factor: int = 2) -> list:
    """Undistorted photos and masks at 1/data_factor resolution with their cameras ("K", and
    "viewmat", world to camera in OpenCV axes), as in stage 4."""
    import pycolmap

    model = pycolmap.Reconstruction(str(Path(stage1) / "sparse" / "0"))
    by_name = {im.name: im for im in model.images.values()}
    views, skipped = [], []
    for p in frames:
        im = by_name.get(Path(p).name)
        if im is None:
            skipped.append(Path(p).name)
            continue
        cam = model.cameras[im.camera_id]
        K = cam.calibration_matrix().copy()
        K[:2] /= data_factor
        dist = opencv_distortion(cam.model.name, cam.params)
        img = cv2.cvtColor(cv2.imread(str(p)), cv2.COLOR_BGR2RGB)
        size = (img.shape[1] // data_factor, img.shape[0] // data_factor)
        img = cv2.undistort(cv2.resize(img, size, interpolation=cv2.INTER_AREA), K, dist)
        m = cv2.resize(masks[Path(p).name].astype(np.float32), size, interpolation=cv2.INTER_AREA)
        viewmat = np.eye(4)
        viewmat[:3] = im.cam_from_world().matrix()
        views.append({"name": Path(p).name, "K": K, "viewmat": viewmat, "image": img,
                      "mask": cv2.undistort(m, K, dist) > 0.5})
    if skipped:  # write_stage1 accepts a model with 80% of the frames registered
        print(f"{len(skipped)} of {len(frames)} frames have no pose in stage1/sparse/0 (stage 1 did not register "
              f"them) and are left out: {', '.join(skipped)}")
    return views


def read_splat_ply(path: Path, device: str = "cuda") -> dict:
    """A 3D Gaussian splat from the standard PLY layout (as stage 3 wrote it): means, log scales,
    rotations, logit opacities and spherical harmonics, as torch tensors."""
    import torch

    with open(path, "rb") as f:
        names, count = [], 0
        while (line := f.readline().decode().strip()) != "end_header":
            if line.startswith("element vertex"):
                count = int(line.split()[-1])
            elif line.startswith("property float"):
                names.append(line.split()[-1])
        data = np.frombuffer(f.read(4 * len(names) * count), dtype="<f4").reshape(count, len(names))
    col = {n: i for i, n in enumerate(names)}
    pick = lambda prefix: data[:, sorted((i for n, i in col.items() if n.startswith(prefix)),
                                         key=lambda i: int(names[i].rsplit("_", 1)[1]))]
    rest = pick("f_rest_")
    t = lambda a: torch.tensor(np.ascontiguousarray(a), dtype=torch.float32, device=device)
    return {"means": t(data[:, [col["x"], col["y"], col["z"]]]), "scales": t(pick("scale_")),
            "quats": t(pick("rot_")), "opacities": t(data[:, col["opacity"]]), "sh0": t(pick("f_dc_"))[:, None],
            "shN": t(rest.reshape(count, 3, -1).transpose(0, 2, 1))}


def render_splat(splat: dict, view: dict, background=(1.0, 1.0, 1.0)) -> np.ndarray:
    """The splat's color image of a view (H x W x 3 in [0, 1]) on a white background."""
    import torch
    from gsplat import rasterization

    h, w = view["image"].shape[:2]
    dev = splat["means"].device
    degree = int(round(np.sqrt(splat["shN"].shape[1] + 1))) - 1
    with torch.no_grad():
        color, _, _ = rasterization(
            splat["means"], torch.nn.functional.normalize(splat["quats"], dim=-1), torch.exp(splat["scales"]),
            torch.sigmoid(splat["opacities"]), torch.cat([splat["sh0"], splat["shN"]], 1),
            torch.tensor(view["viewmat"], dtype=torch.float32, device=dev)[None],
            torch.tensor(view["K"], dtype=torch.float32, device=dev)[None], w, h, sh_degree=degree, packed=False,
            backgrounds=torch.tensor(background, dtype=torch.float32, device=dev)[None])
    return color[0].clamp(0, 1).cpu().numpy()


class MeshRenderer:
    """Rasterizes the textured mesh with nvdiffrast (NVIDIA Source Code License, non-commercial):
    the base color (sRGB), the surface position and normal, and coverage, per pixel."""

    def __init__(self, mesh_uv: dict, base_color: np.ndarray, device: str = "cuda"):
        import nvdiffrast.torch as dr
        import torch

        self.dr, self.ctx = dr, dr.RasterizeCudaContext(device=device)
        t = lambda a, dtype=torch.float32: torch.tensor(np.ascontiguousarray(a), dtype=dtype, device=device)
        self.v, self.f = t(mesh_uv["vertices"]), t(mesh_uv["faces"], torch.int32)
        self.uv, self.tex = t(mesh_uv["uvs"]), t(base_color)[None]
        normals = mesh_uv.get("normals")
        if normals is None:
            import trimesh
            normals = trimesh.Trimesh(mesh_uv["vertices"], mesh_uv["faces"], process=False).vertex_normals
        self.n = t(normals)

    def render(self, K, viewmat, size, near: float = 0.01, far: float = 10.0) -> dict:
        """G-buffer of a pinhole camera K, viewmat (world to camera, OpenCV axes), size (w, h)."""
        import torch

        with torch.no_grad():
            return self._render(K, viewmat, size, near, far)

    def _render(self, K, viewmat, size, near, far):
        import torch

        w, h = size
        K, M = (torch.tensor(np.asarray(a, dtype=float), dtype=torch.float32, device=self.v.device) for a in (K, viewmat))
        cam = self.v @ M[:3, :3].T + M[:3, 3]
        X, Y, Z = cam.unbind(-1)
        clip = torch.stack([(2 * K[0, 0] * X + (2 * K[0, 2] - w) * Z) / w, (2 * K[1, 1] * Y + (2 * K[1, 2] - h) * Z) / h,
                            (far + near) / (far - near) * Z - 2 * far * near / (far - near), Z], -1)
        rast, _ = self.dr.rasterize(self.ctx, clip[None].contiguous(), self.f, (h, w))  # row 0 is the image top
        covered = rast[0, ..., 3] > 0
        uv, _ = self.dr.interpolate(self.uv[None], rast, self.f)
        color = self.dr.texture(self.tex, uv, filter_mode="linear", boundary_mode="clamp")[0]
        position, _ = self.dr.interpolate(self.v[None], rast, self.f)
        normal, _ = self.dr.interpolate(self.n[None], rast, self.f)
        out = {"base_color": color, "position": position[0], "normal": torch.nn.functional.normalize(normal[0], dim=-1),
               "mask": covered}
        return {k: v.cpu().numpy() for k, v in out.items()}


def srgb_to_linear(c):
    c = np.clip(c, 0, 1)
    return np.where(c <= 0.04045, c / 12.92, ((c + 0.055) / 1.055) ** 2.4)


def linear_to_srgb(c):
    c = np.clip(c, 0, 1)
    return np.where(c <= 0.0031308, 12.92 * c, 1.055 * c ** (1 / 2.4) - 0.055)


def masked_psnr(image: np.ndarray, photo: np.ndarray, mask: np.ndarray) -> float:
    """PSNR (dB) over the mask's pixels, for images in [0, 1]."""
    mse = float(((image - photo)[mask] ** 2).mean())
    return float(10 * np.log10(1.0 / max(mse, 1e-12)))


def scoreable_views(views: list, flagged) -> tuple[list, list]:
    """Split views into those the masked scores can use and (name, reason) for the rest: a view whose
    mask is empty has no pixels to score, and a mask the stage 2 drift check flagged is not trusted."""
    kept, left_out = [], []
    for v in views:
        if not v["mask"].any():
            left_out.append((v["name"], "empty mask"))
        elif v["name"] in flagged:
            left_out.append((v["name"], "mask flagged by the stage 2 drift check"))
        else:
            kept.append(v)
    return kept, left_out


def masked_lpips(metric, image: np.ndarray, photo: np.ndarray, mask: np.ndarray, margin: int = 16) -> float:
    """LPIPS between image and photo with everything outside the mask set to white in both, on the
    mask's bounding box plus a margin, so that the score is about the object."""
    ys, xs = np.nonzero(mask)
    y0, y1 = max(ys.min() - margin, 0), min(ys.max() + margin + 1, mask.shape[0])
    x0, x1 = max(xs.min() - margin, 0), min(xs.max() + margin + 1, mask.shape[1])
    crop = lambda a: np.where(mask[..., None], a, 1.0)[y0:y1, x0:x1]
    import torch

    dev = next(metric.parameters()).device
    t = lambda a: torch.tensor(crop(a), dtype=torch.float32, device=dev).permute(2, 0, 1)[None]
    with torch.no_grad():
        return float(metric(t(image), t(photo), normalize=True))


def gain_match(image: np.ndarray, photo: np.ndarray, mask: np.ndarray) -> tuple[np.ndarray, float]:
    """Scale the image's linear intensity by the one gain that best fits the photo on the mask (least
    squares), as an exposure correction; returns the corrected image and the gain."""
    a, b = srgb_to_linear(image)[mask], srgb_to_linear(photo)[mask]
    gain = float((a * b).sum() / max((a * a).sum(), 1e-12))
    return linear_to_srgb(srgb_to_linear(image) * gain), gain

def refined_inputs(stage4: Path, stage4b: Path) -> tuple[dict | None, str | None]:
    """The stage 4b maps ("base_color" RGB, "roughness", "metallic", floats in [0, 1]) and its
    refine.json as {"maps", "record"}, with None as the reason; or None and the reason they are left out.
    The maps are used only when refine.json's mesh_uv_sha256 is the hash of the current
    stage4/mesh_uv.obj: after stage 4 runs again they sit on an older UV layout."""
    import hashlib
    import json

    stage4, stage4b = Path(stage4), Path(stage4b)
    if not stage4b.is_dir():
        return None, f"no {stage4b.name}/ folder: stage 4b has not run in this project, so the refined maps are not scored"
    if not (stage4b / "refine.json").is_file():
        return None, f"{stage4b.name}/ has no refine.json: rerun stage 4b"
    try:
        record = json.loads((stage4b / "refine.json").read_text())
        for key in ("relit", "base_color_matched"):
            float(record["runs"]["refined"][key]["psnr"]), float(record["runs"]["refined"][key]["lpips"])
    except (ValueError, KeyError, TypeError):
        return None, f"{stage4b.name}/refine.json is incomplete or unreadable: rerun stage 4b"
    mesh = stage4 / "mesh_uv.obj"
    current = hashlib.sha256(mesh.read_bytes()).hexdigest() if mesh.is_file() else None
    if record.get("mesh_uv_sha256") != current:
        return None, (f"{stage4.name}/mesh_uv.obj changed after stage 4b ran (stage 4 ran again, or a fallback "
                      f"replaced {stage4.name}/), so the refined maps sit on another UV layout: rerun stage 4b to "
                      "score them")
    maps = {}
    for name in ("base_color", "roughness", "metallic"):
        img = cv2.imread(str(stage4b / f"{name}.png"), cv2.IMREAD_UNCHANGED)
        if img is None:
            return None, f"{stage4b.name}/{name}.png is missing or unreadable: rerun stage 4b"
        maps[name] = (cv2.cvtColor(img, cv2.COLOR_BGR2RGB) if name == "base_color" else img) / 255.0
    return {"maps": maps, "record": record}, None

In [ ]:
import lpips
import torch

frames = sorted((STAGE1 / "frames").glob("*.jpg"))
heldout = set((STAGE1 / "heldout.txt").read_text().split())
drifting = set(json.loads((STAGE2 / "drift.json").read_text())["flagged"])
test = [p for p in frames if p.name in heldout]
train = [p for p in frames if p.name not in heldout and p.name not in drifting]
train_probe = [train[i] for i in np.unique(np.linspace(0, len(train) - 1, len(test)).round().astype(int))]
masks = {p.name: cv2.imread(str(STAGE2 / "masks" / (p.stem + ".png")), cv2.IMREAD_GRAYSCALE) > 127
         for p in test + train_probe}
test_views, heldout_excluded = scoreable_views(load_views(STAGE1, test, masks, data_factor=2), drifting)
train_views, train_excluded = scoreable_views(load_views(STAGE1, train_probe, masks, data_factor=2), drifting)
for split, left_out in (("held-out", heldout_excluded), ("training", train_excluded)):
    if left_out:
        print(f"{len(left_out)} {split} frame(s) left out of the appearance scores: "
              + ", ".join(f"{name} ({why})" for name, why in left_out))
if not test_views:
    raise RuntimeError("no held-out frame has a usable mask, so the appearance cannot be scored: check stage 2")
refined, refined_skipped = refined_inputs(STAGE4, PROJECT / "stage4b")
print("refined maps (stage 4b):", "scored beside the bake" if refined else refined_skipped)
t = time.time()
splat = read_splat_ply(STAGE3 / "splat.ply")
base_color = cv2.cvtColor(cv2.imread(str(STAGE4 / "base_color.png")), cv2.COLOR_BGR2RGB) / 255.0
renderer = MeshRenderer(mesh_uv, base_color)
refined_renderer = MeshRenderer(mesh_uv, refined["maps"]["base_color"]) if refined else None
metric = lpips.LPIPS(net="alex", verbose=False).cuda().eval()
rows, renders = [], {}
for split, views in (("held-out", test_views), ("training", train_views)):
    for v in views:
        size = (v["image"].shape[1], v["image"].shape[0])
        photo, mask = v["image"] / 255.0, v["mask"]
        splat_img = render_splat(splat, v)
        g = renderer.render(v["K"], v["viewmat"], size)
        mesh_img = np.where(g["mask"][..., None], g["base_color"], 1.0)
        matched, gain = gain_match(mesh_img, photo, mask)
        row = {"split": split, "name": v["name"], "gain": gain}
        images = {"splat": splat_img, "mesh": mesh_img, "mesh_matched": matched}
        if refined_renderer is not None:  # the stage 4b base color, scored exactly like the bake
            gr = refined_renderer.render(v["K"], v["viewmat"], size)
            images["refined"] = np.where(gr["mask"][..., None], gr["base_color"], 1.0)
            images["refined_matched"], row["refined_gain"] = gain_match(images["refined"], photo, mask)
        for key, img in images.items():
            row[f"{key}_psnr"], row[f"{key}_lpips"] = masked_psnr(img, photo, mask), masked_lpips(metric, img, photo, mask)
        rows.append(row)
        if split == "held-out":
            renders[v["name"]] = {"photo": photo, "mask": mask, "splat": splat_img, "mesh": mesh_img, "matched": matched}
            if refined_renderer is not None:
                renders[v["name"]] |= {"refined": images["refined"], "refined_matched": images["refined_matched"]}
appearance_s = time.time() - t
mean = lambda split, key: float(np.mean([r[key] for r in rows if r["split"] == split]))
print(f"{len(test_views)} held-out and {len(train_views)} training views at {test_views[0]['image'].shape[1]} x "
      f"{test_views[0]['image'].shape[0]} px, scored in {appearance_s:.0f} s; masked PSNR (dB) and LPIPS (AlexNet), means:")
print(f"{'':28s}{'held-out PSNR':>15s}{'LPIPS':>8s}{'training PSNR':>16s}{'LPIPS':>8s}")
labels = {"splat": "3DGS splat", "mesh": "textured mesh (base color)", "mesh_matched": "textured mesh, gain-matched"}
if refined:
    labels |= {"refined": "refined maps (base color)", "refined_matched": "refined maps, gain-matched"}
for key, label in labels.items():
    print(f"{label:28s}{mean('held-out', key + '_psnr'):15.2f}{mean('held-out', key + '_lpips'):8.3f}"
          f"{mean('training', key + '_psnr'):16.2f}{mean('training', key + '_lpips'):8.3f}")
if refined:
    relit, matched4b = (refined["record"]["runs"]["refined"][k] for k in ("relit", "base_color_matched"))
    print(f"{'refined maps, relit':28s}{relit['psnr']:15.2f}{relit['lpips']:8.3f}   from stage 4b, under its learned "
          f"light; stage 4b's gain-matched score of the same maps: {matched4b['psnr']:.2f} / {matched4b['lpips']:.3f}")
print(f"splat gap, training minus held-out: {mean('training', 'splat_psnr') - mean('held-out', 'splat_psnr'):+.2f} dB PSNR, "
      f"{mean('training', 'splat_lpips') - mean('held-out', 'splat_lpips'):+.3f} LPIPS; "
      f"mesh exposure gain {np.median([r['gain'] for r in rows]):.2f} (median)"
      + (f", refined {np.median([r['refined_gain'] for r in rows]):.2f}" if refined else ""))

In [ ]:
import io

import matplotlib.pyplot as plt
from IPython.display import Image as IPImage, display

scores = {r["name"]: r for r in rows if r["split"] == "held-out"}
names = list(renders)[:: max(1, len(renders) // 3)][:3]
columns = (("photo", None, "held-out photo"), ("splat", "splat", "3DGS splat"), ("mesh", "mesh", "mesh, base color"),
           ("matched", "mesh_matched", "mesh, gain-matched"))
if refined:
    columns += (("refined_matched", "refined_matched", "refined maps, gain-matched"),)
fig, axes = plt.subplots(len(names), len(columns), figsize=(3.25 * len(columns), 3.4 * len(names)), squeeze=False)
for row, name in zip(axes, names):
    r = renders[name]
    ys, xs = np.nonzero(r["mask"])
    box = (slice(max(ys.min() - 20, 0), ys.max() + 20), slice(max(xs.min() - 20, 0), xs.max() + 20))
    for ax, (key, score, title) in zip(row, columns):
        image = r[key] if key == "photo" else np.where(r["mask"][..., None], r[key], 1.0)
        ax.imshow(image[box])
        ax.set_title(f"{title} {name}" if score is None else
                     f"{title}\n{scores[name][score + '_psnr']:.1f} dB, LPIPS {scores[name][score + '_lpips']:.3f}", fontsize=9)
        ax.axis("off")
fig.tight_layout()
buf = io.BytesIO()
fig.savefig(buf, format="jpg", dpi=80)
plt.close(fig)
display(IPImage(data=buf.getvalue()))

## A second light

The appearance scores on held-out frames still use the lighting of the capture. A relightable asset must also look right under a light it was never photographed under. You take one more photo with a lamp moved to a new position, with the board in view, and record the lamp's position in the world frame of stage 1 in `capture/second_light.json` as `{"lamp_m": [x, y, z]}`.

Stage 1's world frame sits on the board. Its origin is the top-left corner of the printed board, marked on the PDF. The $x$ axis runs along the board's top edge to the right, $y$ runs up the page, away from the board across its top edge, and $z$ points up from the table. Leave the board where it lay during the video, or put it back in the same place, and measure the lamp's bulb from that corner with a ruler: its distance along the top edge is $x$, its distance across the top edge is $y$ (negative when the lamp is over the board), and its height above the table is $z$, all in meters.

The photo's camera is found from the board. With the board's square size known, the detected corners fix the focal length (with the principal point at the image center and no lens distortion) and the camera's pose relative to the board, and stage 1's world frame relates the board to everything else. The mesh is then rasterized from that camera, giving at each pixel the base color $a_p$ (converted to linear intensity), the surface point $x_p$ and its normal $n_p$. A diffuse surface lit by an ambient term and a point lamp at $\ell$ reflects

$$
L_p = a_p \left( \alpha + \beta \, \frac{\max(0, \, n_p \cdot \hat{d}_p)}{\lVert \ell - x_p \rVert^2} \right), \qquad \hat{d}_p = \frac{\ell - x_p}{\lVert \ell - x_p \rVert},
$$

which is Lambert's cosine law with the inverse-square falloff of a point source. The lamp's power and the ambient level are unknown, so $\alpha$ and $\beta$ are fitted to the photo by linear least squares over the object's pixels. Everything else, the shading pattern across the object, comes from the geometry, the albedo and the lamp position. The render is scored against the photo with the same PSNR and LPIPS. This model ignores roughness and metallic, so a shiny object's highlights are not reproduced, and the comparison says so in its score. With refined maps present, the same fit runs a second time with the refined base color in place of the bake. Neither albedo was fitted under this lamp, so the two scores show which one carries over to a new light.


In [ ]:
import cv2
import numpy as np


def photo_camera(photo_gray: np.ndarray, board: dict, scale: dict) -> tuple[np.ndarray, np.ndarray]:
    """Intrinsics K and viewmat (world to camera, OpenCV axes) of a still photo that shows the board.

    The focal length comes from the board's own view (principal point fixed at the image center, no
    distortion), and the pose from the detected corners. The world frame is stage 1's: the board's
    corners in it are in scale.json ("board_corners_world")."""
    sq, mk = board["square_mm_measured"] / 1000, board["marker_mm_measured"] / 1000
    dictionary = cv2.aruco.getPredefinedDictionary(getattr(cv2.aruco, board["dictionary"]))
    charuco = cv2.aruco.CharucoBoard((board["squares_x"], board["squares_y"]), sq, mk, dictionary)
    corners, ids, _, _ = cv2.aruco.CharucoDetector(charuco).detectBoard(photo_gray)
    if ids is None or len(ids) < 8:
        raise RuntimeError(f"the board shows {0 if ids is None else len(ids)} corners in the second-light photo; "
                           "at least 8 are needed")
    obj = np.asarray(charuco.getChessboardCorners(), dtype=np.float32)[ids.ravel()]
    h, w = photo_gray.shape
    K0 = np.array([[max(w, h), 0, w / 2], [0, max(w, h), h / 2], [0, 0, 1]], dtype=float)
    flags = (cv2.CALIB_USE_INTRINSIC_GUESS | cv2.CALIB_FIX_PRINCIPAL_POINT | cv2.CALIB_FIX_ASPECT_RATIO
             | cv2.CALIB_ZERO_TANGENT_DIST | cv2.CALIB_FIX_K1 | cv2.CALIB_FIX_K2 | cv2.CALIB_FIX_K3)
    _, K, _, rvecs, tvecs = cv2.calibrateCamera([obj], [corners.reshape(-1, 1, 2).astype(np.float32)], (w, h), K0,
                                                np.zeros(5), flags=flags)
    cam_from_board = np.eye(4)
    cam_from_board[:3, :3], cam_from_board[:3, 3] = cv2.Rodrigues(rvecs[0])[0], tvecs[0].ravel()
    # world_from_board: the rigid fit of the board's own corners to their world positions in scale.json
    src = np.asarray(charuco.getChessboardCorners(), dtype=float)
    if "board_corners_world" not in scale:
        raise RuntimeError("stage1/scale.json has no board_corners_world: the second-light camera needs the board's "
                           "corners in the world frame, which scale_from_board returns and write_stage1 keeps. "
                           "Run stage 1 again on your capture.")
    dst = np.asarray(scale["board_corners_world"], dtype=float)
    if dst.shape != src.shape:
        raise RuntimeError(f"stage1/scale.json holds {len(dst)} board corners, the board has {len(src)}: "
                           "scale_from_board must return every chessboard corner, in getChessboardCorners() order")
    mu_s, mu_d = src.mean(0), dst.mean(0)
    U, _, Vt = np.linalg.svd((dst - mu_d).T @ (src - mu_s))
    R = U @ np.diag([1.0, 1.0, np.sign(np.linalg.det(U @ Vt))]) @ Vt
    world_from_board = np.eye(4)
    world_from_board[:3, :3], world_from_board[:3, 3] = R, mu_d - R @ mu_s
    return K, cam_from_board @ np.linalg.inv(world_from_board)


def lamp_terms(gbuffer: dict, lamp_m) -> np.ndarray:
    """Per-pixel irradiance from a point lamp of unit power, Lambert's cosine over the squared distance."""
    to_lamp = np.asarray(lamp_m, dtype=float) - gbuffer["position"]
    d2 = np.maximum((to_lamp ** 2).sum(-1), 1e-6)
    cosine = np.clip((gbuffer["normal"] * to_lamp).sum(-1) / np.sqrt(d2), 0, None)
    return cosine / d2


def fit_light(albedo_linear: np.ndarray, irradiance: np.ndarray, photo_linear: np.ndarray, mask: np.ndarray):
    """Least-squares ambient a and lamp strength b >= 0 in photo = albedo (a + b irradiance), on the mask."""
    A = np.stack([albedo_linear[mask].ravel(), (albedo_linear * irradiance[..., None])[mask].ravel()], 1)
    (a, b), *_ = np.linalg.lstsq(A, photo_linear[mask].ravel(), rcond=None)
    return max(float(a), 0.0), max(float(b), 0.0)


def relight(albedo_linear: np.ndarray, irradiance: np.ndarray, a: float, b: float) -> np.ndarray:
    """The diffuse render under the fitted light, in sRGB."""
    return linear_to_srgb(albedo_linear * (a + b * irradiance[..., None]))

def second_light_score(renderer, K, viewmat, photo: np.ndarray, lamp_m, metric) -> dict:
    """Render a base color from the photo's camera, fit the ambient and lamp terms to the photo and
    score the relit render on the rendered object's pixels: masked PSNR and LPIPS, ambient, lamp."""
    g = renderer.render(K, viewmat, (photo.shape[1], photo.shape[0]))
    mask = g["mask"]
    albedo = srgb_to_linear(g["base_color"])
    E = lamp_terms(g, lamp_m)
    a, b = fit_light(albedo, E, srgb_to_linear(photo), mask)
    relit = np.where(mask[..., None], relight(albedo, E, a, b), 1.0)
    return {"psnr_db": masked_psnr(relit, photo, mask), "lpips": masked_lpips(metric, relit, photo, mask),
            "ambient": a, "lamp_strength": b}

In [ ]:
second_light_photo = next((p for p in (CAPTURE / "second_light.jpg", CAPTURE / "second_light.png") if p.exists()), None)
second_light, second_light_refined = None, None
if second_light_photo is None or not (CAPTURE / "second_light.json").exists():
    second_light_reason = ("the reference dataset has no photo under a second lamp, so the comparison is skipped"
                           if REFERENCE else "capture/second_light.jpg or second_light.json is missing")
    print("second light:", second_light_reason)
else:
    lamp = json.loads((CAPTURE / "second_light.json").read_text())
    board = json.loads((CAPTURE / "board.json").read_text())
    photo_bgr = cv2.imread(str(second_light_photo))
    K2, viewmat2 = photo_camera(cv2.cvtColor(photo_bgr, cv2.COLOR_BGR2GRAY), board, scale)
    photo2 = cv2.cvtColor(photo_bgr, cv2.COLOR_BGR2RGB) / 255.0
    second_light = second_light_score(renderer, K2, viewmat2, photo2, lamp["lamp_m"], metric)
    second_light |= {"lamp_m": lamp["lamp_m"], "photo": second_light_photo.name}
    if refined_renderer is not None:
        second_light_refined = second_light_score(refined_renderer, K2, viewmat2, photo2, lamp["lamp_m"], metric)
    second_light_reason = None
    for label, s in (("bake", second_light), ("refined maps", second_light_refined)):
        if s is not None:
            print(f"second light at {lamp['lamp_m']} m, {label}: fitted ambient {s['ambient']:.3f}, lamp "
                  f"{s['lamp_strength']:.4f}; masked PSNR {s['psnr_db']:.2f} dB, LPIPS {s['lpips']:.3f} against your photo")

## Generated against measured

Every stage so far measured the object, from many photos, with a metric scale from stage 1 and a surface fused from depth that many views agree on. A generative model skips the measurement. TRELLIS.2 (Xiang et al., "Native and Compact Structured Latents for 3D Generation", arXiv 2512.14692) turns one photo into a textured mesh with base color, roughness and metallic maps. This section scores its output on the two reference objects with the same tools that scored the measured twin.

The generator conditions on one masked photo. For each reference object the course used the training frame with the largest stage 2 mask and passed that mask as the photo's alpha channel, so the generator saw the same segmentation as the measured pipeline. Everything the photo does not show comes from what the model learned: the back and the base, the depth of the object along the line of sight, and the materials, which one photo under one light does not separate from the lighting. Its output has no unit; it lies in a box of side 1. The model was trained on Objaverse-XL (Deitke et al., "Objaverse-XL: A Universe of 10M+ 3D Objects", NeurIPS 2023). Whether that collection holds the OmniObject3D objects used here is unknown, so part of a good score could be recall of a similar training asset.

TRELLIS.2's code and weights are under the MIT license. It encodes the photo with Meta's DINOv3, whose weights carry the DINOv3 License, and no output exists without them. TRELLIS.2's authors list a GPU with at least 24 GB of memory as a requirement, so the course ran it once for each object on a 96 GB card, and you read the results from `generated/<object>/`. The second object, the hammer, is a robustness object: its measured pipeline does not pass every check that was calibrated on the carton, so its measured rows are a harder case, not a reference. If you want a generated version of your own object, you may run the public TRELLIS.2 demo on the Hugging Face Space `microsoft/TRELLIS.2` with one of your frames. This is optional and not graded, and the demo sends your photo to a third-party service.

The generated mesh gets its scale from the measured one. `scale_to_metric` divides the longest side of the measured mesh's oriented bounding box by the longest side of the generated mesh's. The measured mesh is in meters through stage 1, so after scaling the generated object has the measured length along its longest side, and its other two sides are the generator's own proportions. Comparing those two sides with the measured ones is the first test of what it invented.

The generated mesh also has its own orientation. After scaling, each mesh's oriented bounding box is set on the other's in each of the 24 ways a box can rest on a box. These are the rotations that permute the three axes with signs, the signed permutation matrices with determinant $+1$. Iterative closest points then refines each start with a rotation and a translation, never a reflection: a mirror image of an object is a different object, with its print reversed. The start that ends with the lowest root mean square distance (RMSE) to the measured vertices wins. A box fits itself equally well in four poses, so the distinct poses that end within 10% of the best RMSE are compared once more, by how well the generated texture, seen from the condition frame's camera, agrees with the condition photo. The comparison records every tied pose with its RMSE and its disagreement with the photo. The check fails a comparison whose RMSE exceeds 5% of the measured mesh's longest side, because every later number would then compare different surfaces.

Both meshes are then measured against the dataset's scan. The scan is placed on the measured stage 3 mesh once, as in stage 3, and that one placement serves both Chamfer distances, so the two numbers differ only in the meshes. The placement is fitted to the measured mesh, so if it favors either mesh, it favors the measured one. Each mesh's accuracy, the mean distance from its surface to the scan, is also split into the surface the condition frame saw and the rest. For the generated mesh the rest is invented; for the measured mesh, which many views saw, the same split is a control. The held-out frames are rendered from both meshes with their own base color, unlit and gain-matched, as the bake was scored above, and the measured mesh's relit score is copied from stage 4b. The cell also prints the generated object's mean metallic and roughness next to the stage 4 bake's, and how many box poses tied and which one the photo chose.

Both generated objects come out with metallic 0 on every texel. On the carton the course checked that these zeros are TRELLIS.2's own near-zero prediction and not negative values clipped when the maps were saved; the hammer was not checked. Training assets without a material get metallic 0, which likely pulls the model toward 0.

The viewer shows each object under `generated/<object>`, side by side from left to right: the aligned generated mesh with its texture, the same mesh colored by distance to the scan, the measured stage 4 mesh colored the same way, and the placed scan. The colors run from dark purple at 0 mm to yellow at 10 mm or more.

In [ ]:
import itertools
import json
from pathlib import Path

import cv2
import numpy as np
import open3d as o3d
from scipy.spatial import cKDTree

ALIGN_LIMIT_SHARE = 0.05  # an alignment RMSE above 5% of the measured mesh's longest side fails
TIE_SHARE = 1.10  # starts that end within 10% of the best RMSE fit the shape equally well


def chamfer_mm(mesh, scan_path: Path, scan_from_world, n: int = 200_000, seed: int = 0) -> dict:
    """Symmetric Chamfer distance between the mesh (meters) and the scan (millimeters, inferred).

    Both surfaces are sampled uniformly with n points, and the scan is brought into the world
    frame with the inverse of scan_from_world. Accuracy is the mean distance from mesh points to
    the scan, completeness from scan points to the mesh, and the Chamfer distance their mean.
    """
    o3d.utility.random.seed(seed)
    scan = o3d.io.read_triangle_mesh(str(scan_path))
    scan.transform(np.linalg.inv(np.asarray(scan_from_world)))  # millimeters to world meters
    a = np.asarray(mesh.sample_points_uniformly(n).points)
    b = np.asarray(scan.sample_points_uniformly(n).points)
    acc = cKDTree(b).query(a)[0] * 1000
    comp = cKDTree(a).query(b)[0] * 1000
    return {"accuracy_mm": float(acc.mean()), "completeness_mm": float(comp.mean()),
            "chamfer_mm": float((acc.mean() + comp.mean()) / 2)}


def register_scan(mesh, scan_path: Path, scan_from_world, n: int = 200_000, seed: int = 0,
                  max_distance_m: float = 0.02) -> dict:
    """Re-register the scan to the mesh, starting from the stage 1 placement.

    The scan's points are moved so their centroid matches the mesh's, then point-to-point ICP with
    a scale factor fits them to the mesh. Returns offset_mm (how far the scan's center moved from
    its stage 1 placement), scale, rotation_deg, and scan_from_world for the new placement, which
    chamfer_mm accepts.
    """
    o3d.utility.random.seed(seed)
    scan = o3d.io.read_triangle_mesh(str(scan_path))
    world_from_scan = np.linalg.inv(np.asarray(scan_from_world, dtype=float))
    scan.transform(world_from_scan)
    a = mesh.sample_points_uniformly(n)
    b = scan.sample_points_uniformly(n)
    center = np.asarray(b.points).mean(0)
    init = np.eye(4)
    init[:3, 3] = np.asarray(a.points).mean(0) - center
    reg = o3d.pipelines.registration.registration_icp(
        b, a, max_distance_m, init,
        o3d.pipelines.registration.TransformationEstimationPointToPoint(with_scaling=True),
        o3d.pipelines.registration.ICPConvergenceCriteria(max_iteration=200))
    T = np.asarray(reg.transformation)
    scale = float(np.cbrt(np.linalg.det(T[:3, :3])))
    turn = np.degrees(np.arccos(np.clip((np.trace(T[:3, :3] / scale) - 1) / 2, -1, 1)))
    return {"offset_mm": float(np.linalg.norm(T[:3, :3] @ center + T[:3, 3] - center) * 1000),
            "scale": scale, "rotation_deg": float(turn), "fitness": float(reg.fitness),
            "scan_from_world": np.linalg.inv(T @ world_from_scan)}

def proper_axis_rotations() -> list:
    """The 24 rotations that map the coordinate axes onto themselves: signed permutation matrices
    with determinant +1. The 24 with determinant -1 are mirror images and are never tried."""
    rotations = []
    for perm in itertools.permutations(range(3)):
        for signs in itertools.product((1.0, -1.0), repeat=3):
            R = np.zeros((3, 3))
            R[range(3), perm] = signs
            if np.linalg.det(R) > 0:
                rotations.append(R)
    return rotations


def alignment_candidates(generated_vertices, measured_vertices, scale: float, n: int = 4000, seed: int = 0) -> list:
    """Rigid ICP of the scaled generated vertices onto the measured ones, from each of 24 starts.

    Each start sets the generated mesh's oriented bounding box on the measured mesh's, center on
    center, under one of the 24 axis rotations, so every way of laying one box on the other is
    tried once. ICP (trimesh) then refines rotation and translation only: no reflection, no scale.
    Returns (T, rmse_mm) for each start, best first. T is the 4 x 4 world-from-generated transform
    with the scale applied; rmse_mm is the root mean square distance from the moved generated
    points to the nearest measured vertex, in millimeters.
    """
    import trimesh

    rng = np.random.default_rng(seed)
    g = np.asarray(generated_vertices, dtype=float) * scale
    m = np.asarray(measured_vertices, dtype=float)
    pick = lambda a, k: a[rng.choice(len(a), min(k, len(a)), replace=False)]
    g_sample, m_sample = pick(g, n), pick(m, 5 * n)
    to_g, _ = trimesh.bounds.oriented_bounds(g)
    to_m, _ = trimesh.bounds.oriented_bounds(m)
    tree = cKDTree(m)
    S = np.diag([scale, scale, scale, 1.0])
    out = []
    for R in proper_axis_rotations():
        start = np.eye(4)
        start[:3, :3] = R
        M, moved, _ = trimesh.registration.icp(g_sample, m_sample, initial=np.linalg.inv(to_m) @ start @ to_g,
                                               threshold=1e-10, max_iterations=60, reflection=False, scale=False)
        out.append((M @ S, float(np.sqrt((tree.query(moved)[0] ** 2).mean()) * 1000)))
    return sorted(out, key=lambda c: c[1])


def turn_deg(A, B) -> float:
    """The angle (degrees) of the rotation between the rotation parts of two transforms, scale removed."""
    Ra, Rb = (np.asarray(T, dtype=float)[:3, :3] / np.cbrt(np.linalg.det(np.asarray(T, dtype=float)[:3, :3])) for T in (A, B))
    return float(np.degrees(np.arccos(np.clip((np.trace(Ra @ Rb.T) - 1) / 2, -1, 1))))


def align_generated(generated_vertices, measured_vertices, scale: float, prefer=None,
                    ties: list | None = None) -> tuple[np.ndarray, float]:
    """The 4 x 4 world-from-generated transform (scale applied) and its RMSE in millimeters.

    Of the 24 starts of alignment_candidates, the distinct poses (starts that converged within 2
    degrees of each other count once) that end within TIE_SHARE of the best RMSE fit the shape
    equally well, as the faces of a box do. prefer(T), when given, scores each of them (lower is
    better) and picks one; otherwise, or when only one pose is in the band, the lowest RMSE wins.
    ties, when a list is given, receives one record per pose in the band: "rmse_mm", "photo_disagreement"
    (prefer's score, None when prefer did not run), "turn_deg" (from the chosen pose), "lowest_rmse"
    and "chosen".
    """
    candidates = alignment_candidates(generated_vertices, measured_vertices, scale)
    poses = []
    for T, rmse in candidates:
        if rmse <= TIE_SHARE * candidates[0][1] and all(turn_deg(T, P) > 2.0 for P, _ in poses):
            poses.append((T, rmse))
    scores = [prefer(T) for T, _ in poses] if prefer is not None and len(poses) > 1 else [None] * len(poses)
    k = int(np.argmin(scores)) if scores[0] is not None else 0
    T, rmse = poses[k]
    assert np.linalg.det(T[:3, :3]) > 0, "an alignment must never mirror the generated mesh"
    if ties is not None:
        ties[:] = [{"rmse_mm": r, "photo_disagreement": None if s is None else float(s), "turn_deg": turn_deg(P, T),
                    "lowest_rmse": i == 0, "chosen": i == k} for i, ((P, r), s) in enumerate(zip(poses, scores))]
    return T, rmse


def transform_mesh_uv(mesh_uv: dict, T) -> dict:
    """The mesh moved by T (a rotation times a scale, then a translation); normals turn with it."""
    T = np.asarray(T, dtype=float)
    out = dict(mesh_uv, vertices=np.asarray(mesh_uv["vertices"], dtype=float) @ T[:3, :3].T + T[:3, 3])
    if mesh_uv.get("normals") is not None:
        R = T[:3, :3] / np.cbrt(np.linalg.det(T[:3, :3]))
        out["normals"] = np.asarray(mesh_uv["normals"], dtype=float) @ R.T
    return out


def surface_samples(mesh: dict, n: int, seed: int = 0) -> dict:
    """n points spread uniformly over the mesh's area ("points"), with their texture coordinates
    ("uvs", None when the mesh has none)."""
    import trimesh

    tm = trimesh.Trimesh(mesh["vertices"], mesh["faces"], process=False)
    points, face = trimesh.sample.sample_surface(tm, n, seed=seed)
    uvs = None
    if mesh.get("uvs") is not None:
        bary = trimesh.triangles.points_to_barycentric(tm.triangles[face], points)
        uvs = (np.asarray(mesh["uvs"])[np.asarray(mesh["faces"])[face]] * bary[..., None]).sum(1)
    return {"points": np.asarray(points), "uvs": uvs}


def texel(image: np.ndarray, uvs: np.ndarray) -> np.ndarray:
    """The texel of image under each texture coordinate (origin at the top left, v down)."""
    h, w = image.shape[:2]
    return image[np.clip((uvs[:, 1] * h).astype(int), 0, h - 1), np.clip((uvs[:, 0] * w).astype(int), 0, w - 1)]


def seen_from(vertices, faces, points, K, viewmat, size, tol: float = 0.002) -> np.ndarray:
    """Which surface points a camera sees: in front of it, inside the w x h image, and not hidden
    by the mesh (the first surface along the ray from the camera center is the point itself,
    within tol meters)."""
    points, viewmat, K = np.asarray(points, dtype=float), np.asarray(viewmat, dtype=float), np.asarray(K, dtype=float)
    scene = o3d.t.geometry.RaycastingScene()
    scene.add_triangles(o3d.core.Tensor(np.asarray(vertices, dtype=np.float32)),
                        o3d.core.Tensor(np.asarray(faces, dtype=np.uint32)))
    center = -viewmat[:3, :3].T @ viewmat[:3, 3]
    ray = points - center
    dist = np.linalg.norm(ray, axis=1)
    rays = np.c_[np.broadcast_to(center, points.shape), ray / dist[:, None]].astype(np.float32)
    hit = scene.cast_rays(o3d.core.Tensor(rays))["t_hit"].numpy()
    cam = points @ viewmat[:3, :3].T + viewmat[:3, 3]
    px = cam @ K.T
    z = np.where(cam[:, 2] > 1e-9, px[:, 2], np.inf)
    x, y = px[:, 0] / z, px[:, 1] / z
    w, h = size
    return (cam[:, 2] > 1e-9) & (x >= 0) & (x < w) & (y >= 0) & (y < h) & (hit >= dist - tol)


def photo_disagreement(generated: dict, base_color: np.ndarray, T, view: dict, n: int = 50_000, seed: int = 0) -> float:
    """How far the generated texture, placed by T and seen from a view's camera, is from that photo:
    the mean absolute sRGB difference over the visible surface points that land on the object's
    mask, after one exposure gain (as gain_match). Lower is better."""
    placed = transform_mesh_uv(generated, T)
    s = surface_samples(placed, n, seed)
    h, w = view["image"].shape[:2]
    seen = seen_from(placed["vertices"], placed["faces"], s["points"], view["K"], view["viewmat"], (w, h))
    cam = s["points"][seen] @ np.asarray(view["viewmat"])[:3, :3].T + np.asarray(view["viewmat"])[:3, 3]
    px = cam @ np.asarray(view["K"]).T
    x, y = (px[:, 0] / px[:, 2]).astype(int), (px[:, 1] / px[:, 2]).astype(int)
    on = view["mask"][y, x]
    if not on.any():
        return float("inf")
    a = srgb_to_linear(texel(base_color, s["uvs"][seen][on]))
    b = srgb_to_linear(view["image"][y[on], x[on]] / 255.0)
    gain = float((a * b).sum() / max((a * a).sum(), 1e-12))
    return float(np.abs(linear_to_srgb(a * gain) - linear_to_srgb(b)).mean())


def scan_points(scan_path: Path, scan_from_world, n: int = 200_000, seed: int = 0) -> np.ndarray:
    """n points sampled on the scan, brought into the world frame as chamfer_mm does."""
    o3d.utility.random.seed(seed)
    scan = o3d.io.read_triangle_mesh(str(scan_path))
    scan.transform(np.linalg.inv(np.asarray(scan_from_world)))
    return np.asarray(scan.sample_points_uniformly(n).points)


def measure_generated(generated: dict, measured_mesh, scan_path: Path, scan_from_world, condition_view: dict | None = None,
                      base_color: np.ndarray | None = None, n: int = 200_000, seed: int = 0) -> dict:
    """Scale and align the generated mesh on the measured one and measure both against the scan.

    generated is the TRELLIS.2 mesh as load_mesh_uv reads it (its own unit and frame), and
    measured_mesh the stage 3 mesh (Open3D, meters). The scale is scale_to_metric's and the
    alignment align_generated's; with condition_view and base_color, ties are broken by
    photo_disagreement with the condition photo, and "alignment_ties" records every tied pose. The scan is placed once, by register_scan on the
    measured mesh, and that one placement serves both Chamfer distances. With condition_view the
    accuracy of each mesh (mean distance from its surface to the scan) is also split into the part
    the condition frame saw and the part it did not.
    """
    import trimesh

    measured_v = np.asarray(measured_mesh.vertices)
    scale = scale_to_metric(generated["vertices"], measured_v)
    prefer = None
    if condition_view is not None and base_color is not None:
        prefer = lambda T: photo_disagreement(generated, base_color, T, condition_view)
    ties = []
    T, rmse = align_generated(generated["vertices"], measured_v, scale, prefer=prefer, ties=ties)
    placed = transform_mesh_uv(generated, T)
    placed_o3d = o3d.geometry.TriangleMesh(o3d.utility.Vector3dVector(placed["vertices"]),
                                           o3d.utility.Vector3iVector(np.asarray(placed["faces"], dtype=np.int32)))
    placement = register_scan(measured_mesh, scan_path, scan_from_world, n=n, seed=seed)
    chamfer = {"generated": chamfer_mm(placed_o3d, scan_path, placement["scan_from_world"], n=n, seed=seed),
               "measured": chamfer_mm(measured_mesh, scan_path, placement["scan_from_world"], n=n, seed=seed)}
    extents = lambda v: sorted(float(x) * 1000 for x in trimesh.bounds.oriented_bounds(np.asarray(v))[1])
    out = {"scale": float(scale), "T": T, "align_rmse_mm": float(rmse), "alignment_ties": ties,
           "measured_extent_mm": max(extents(measured_v)),
           "extents_mm": {"generated": extents(placed["vertices"]), "measured": extents(measured_v)},
           "chamfer_mm": {k: c["chamfer_mm"] for k, c in chamfer.items()},
           "chamfer_parts_mm": {k: {"accuracy": c["accuracy_mm"], "completeness": c["completeness_mm"]}
                                for k, c in chamfer.items()},
           "scan_placement": {k: placement[k] for k in ("offset_mm", "scale", "rotation_deg")},
           "scan_from_world": placement["scan_from_world"], "placed": placed,
           "accuracy_mm": None, "seen_fraction": None, "scan_points": None}
    if condition_view is not None:
        scan_pts = scan_points(scan_path, placement["scan_from_world"], n, seed)
        tree = cKDTree(scan_pts)
        size = (condition_view["image"].shape[1], condition_view["image"].shape[0])
        measured = {"vertices": measured_v, "faces": np.asarray(measured_mesh.triangles)}
        out["accuracy_mm"] = {}
        for name, mesh in (("generated", placed), ("measured", measured)):
            points = surface_samples(mesh, n // 4, seed)["points"]
            seen = seen_from(mesh["vertices"], mesh["faces"], points, condition_view["K"], condition_view["viewmat"], size)
            d = tree.query(points)[0] * 1000
            out["accuracy_mm"][name] = {"seen": float(d[seen].mean()), "unseen": float(d[~seen].mean())}
            if name == "generated":
                out["seen_fraction"] = float(seen.mean())
        out["scan_points"] = scan_pts
    return out


def score_mesh(renderer, views: list, metric) -> dict:
    """Masked PSNR and LPIPS of a mesh's base color on each view, unlit and after one exposure gain
    (gain_match), as the bake is scored above: the means as "unlit" and "matched" ({"psnr",
    "lpips"}), the per-frame values, and the gain-matched renders."""
    rows, renders = [], {}
    for v in views:
        photo, mask = v["image"] / 255.0, v["mask"]
        g = renderer.render(v["K"], v["viewmat"], (photo.shape[1], photo.shape[0]))
        image = np.where(g["mask"][..., None], g["base_color"], 1.0)
        matched, gain = gain_match(image, photo, mask)
        rows.append({"name": v["name"], "gain": gain,
                     "unlit": {"psnr": masked_psnr(image, photo, mask), "lpips": masked_lpips(metric, image, photo, mask)},
                     "matched": {"psnr": masked_psnr(matched, photo, mask),
                                 "lpips": masked_lpips(metric, matched, photo, mask)}})
        renders[v["name"]] = matched
    mean = lambda key: {s: float(np.mean([r[key][s] for r in rows])) for s in ("psnr", "lpips")}
    return {"unlit": mean("unlit"), "matched": mean("matched"), "per_frame": rows, "renders": renders}


def compare_generated(folder: Path, lpips_metric, extras: dict | None = None) -> dict:
    """One entry of generated_vs_measured.json, from a folder that ensure_generated filled.

    The folder holds a reference object's measured stages, its ground truth and trellis2/. The
    generated mesh is scaled, aligned and measured by measure_generated. The held-out frames are
    rendered from both meshes, each with its own base color, unlit and gain-matched; the measured
    mesh's relit score is copied from stage4b/refine.json. extras, when given, receives what the
    figure and the viewer draw: the placed meshes, their textures, the scan points and the renders.
    """
    folder = Path(folder)
    record = json.loads((folder / "trellis2" / "trellis2.json").read_text())
    scan_from_world = json.loads((folder / "stage1" / "scale.json").read_text())["scan_from_world"]
    read_rgb = lambda p: cv2.cvtColor(cv2.imread(str(p)), cv2.COLOR_BGR2RGB) / 255.0
    read_gray = lambda p: cv2.imread(str(p), cv2.IMREAD_GRAYSCALE) / 255.0
    generated = load_mesh_uv(folder / "trellis2" / "mesh_uv.obj")
    generated_color = read_rgb(folder / "trellis2" / "base_color.png")
    measured_uv = load_mesh_uv(folder / "stage4" / "mesh_uv.obj")
    measured_color = read_rgb(folder / "stage4" / "base_color.png")
    heldout = (folder / "stage1" / "heldout.txt").read_text().split()
    flagged = set(json.loads((folder / "stage2" / "drift.json").read_text())["flagged"])
    names = heldout + [record["condition_frame"]]
    masks = {n: cv2.imread(str(folder / "stage2" / "masks" / (Path(n).stem + ".png")), cv2.IMREAD_GRAYSCALE) > 127
             for n in names}
    frames = folder / "stage1" / "frames"
    views, left_out = scoreable_views(load_views(folder / "stage1", [frames / n for n in heldout], masks), flagged)
    condition = load_views(folder / "stage1", [frames / record["condition_frame"]], masks)[0]
    geo = measure_generated(generated, o3d.io.read_triangle_mesh(str(folder / "stage3" / "mesh.ply")),
                            folder / "ground_truth" / "scan" / "Scan.obj", scan_from_world,
                            condition_view=condition, base_color=generated_color)
    scores = {"generated": score_mesh(MeshRenderer(geo["placed"], generated_color), views, lpips_metric),
              "measured": score_mesh(MeshRenderer(measured_uv, measured_color), views, lpips_metric)}
    relit = json.loads((folder / "stage4b" / "refine.json").read_text())["runs"]["refined"]["relit"]
    surface_mean = lambda mesh, image: float(texel(image, surface_samples(mesh, 100_000)["uvs"]).mean())
    materials = {}
    for name, mesh, maps in (("generated", geo["placed"], folder / "trellis2"), ("measured", measured_uv, folder / "stage4")):
        materials[name] = {f"{k}_mean": surface_mean(mesh, read_gray(maps / f"{k}.png")) for k in ("metallic", "roughness")}
    entry = {
        "object": folder.name, "condition_frame": record["condition_frame"], "scale": geo["scale"],
        "align_rmse_mm": geo["align_rmse_mm"], "alignment_ties": geo["alignment_ties"],
        "measured_extent_mm": geo["measured_extent_mm"],
        "extents_mm": geo["extents_mm"], "chamfer_mm": geo["chamfer_mm"], "chamfer_parts_mm": geo["chamfer_parts_mm"],
        "scan_placement": geo["scan_placement"], "accuracy_mm": geo["accuracy_mm"], "seen_fraction": geo["seen_fraction"],
        "held_out": {"generated": scores["generated"]["unlit"], "generated_matched": scores["generated"]["matched"],
                     "measured": scores["measured"]["unlit"], "measured_matched": scores["measured"]["matched"],
                     "measured_relit": {"psnr": float(relit["psnr"]), "lpips": float(relit["lpips"])}},
        "heldout_frames": [v["name"] for v in views],
        "heldout_excluded": [{"name": n, "reason": why} for n, why in left_out],
        "held_out_per_frame": [{"name": g["name"], "generated_matched": g["matched"], "measured_matched": m["matched"]}
                               for g, m in zip(scores["generated"]["per_frame"], scores["measured"]["per_frame"])],
        "materials": materials,
        "trellis2": {k: record[k] for k in ("repo_commit", "model_revision", "pipeline_type", "seed")},
    }
    if extras is not None:
        extras.update(placed=geo["placed"], generated_color=generated_color, measured_uv=measured_uv,
                      measured_color=measured_color, scan_points=geo["scan_points"],
                      renders={v["name"]: {"photo": v["image"] / 255.0, "mask": v["mask"],
                                           "generated": scores["generated"]["renders"][v["name"]],
                                           "measured": scores["measured"]["renders"][v["name"]]} for v in views})
    return entry


def generated_problems(entry: dict) -> list[str]:
    """Failures of one comparison. An alignment RMSE above ALIGN_LIMIT_SHARE of the measured mesh's
    longest side means the generated mesh sits upside down, on the wrong face or off the object,
    and every number after it would compare different surfaces. Fewer than 3 scored held-out
    frames leave the photo scores without support."""
    problems = []
    name = entry.get("object", "the generated object")
    limit = ALIGN_LIMIT_SHARE * entry["measured_extent_mm"]
    if not entry["align_rmse_mm"] <= limit:  # also catches nan
        problems.append(f"{name}: the alignment leaves {entry['align_rmse_mm']:.1f} mm RMSE to the measured mesh, over "
                        f"{limit:.1f} mm ({100 * ALIGN_LIMIT_SHARE:.0f}% of its {entry['measured_extent_mm']:.0f} mm "
                        "longest side), so the generated mesh is not lying on the measured one")
    if "heldout_frames" in entry and len(entry["heldout_frames"]) < 3:
        problems.append(f"{name}: only {len(entry['heldout_frames'])} held-out frames could be scored (at least 3)")
    return problems


def generated_notes(entry: dict) -> list[str]:
    """What the comparison shows the generator could not know, stated from the entry's numbers."""
    notes, name = [], entry.get("object", "object")
    materials = entry.get("materials")
    if materials and materials["generated"]["metallic_mean"] < 0.005:
        notes.append(f"metallic is 0 on every texel of the generated {name}: TRELLIS.2 made the whole object a "
                     f"dielectric (roughness mean {materials['generated']['roughness_mean']:.2f}); the stage 4 bake "
                     f"of the measured mesh has metallic mean {materials['measured']['metallic_mean']:.2f}")
    extents = entry.get("extents_mm")
    if extents:
        g, m = sorted(extents["generated"]), sorted(extents["measured"])
        notes.append(f"scaled by its longest side ({m[2]:.0f} mm), the generated object's other sides are {g[0]:.1f} "
                     f"and {g[1]:.1f} mm against {m[0]:.1f} and {m[1]:.1f} mm measured "
                     f"({100 * (g[0] / m[0] - 1):+.1f}%, {100 * (g[1] / m[1] - 1):+.1f}%)")
    accuracy = entry.get("accuracy_mm")
    if accuracy:
        notes.append(f"the {100 * (1 - entry['seen_fraction']):.0f}% of the generated surface that the condition frame "
                     f"did not show lies {accuracy['generated']['unseen']:.1f} mm from the scan on average, against "
                     f"{accuracy['generated']['seen']:.1f} mm for the part it showed (the measured mesh: "
                     f"{accuracy['measured']['unseen']:.1f} and {accuracy['measured']['seen']:.1f} mm on its own "
                     "unseen and seen parts)")
    return notes

In [ ]:
import numpy as np


def scale_to_metric(generated_vertices: np.ndarray, measured_vertices: np.ndarray) -> float:
    """Meters per unit of the generated mesh: the longest side of the measured mesh's oriented
    bounding box, which is in meters through stage 1, over the longest side of the generated
    mesh's oriented bounding box.

    trimesh.bounds.oriented_bounds(points) returns the transform into the box's frame and the
    box's three side lengths. Both arguments are (N, 3) vertex arrays.
    """
    raise NotImplementedError

In [ ]:
t = time.time()
comparisons, generated_extras = [], {}
for obj in GENERATED_OBJECTS:
    folder = ensure_generated(obj)  # generated/<obj>/: that object's measured stages, its scan and trellis2/
    extras = {}
    entry = compare_generated(folder, metric, extras)
    comparisons.append(entry)
    generated_extras[obj] = extras
    held = entry["held_out"]
    print(f"{obj}: condition frame {entry['condition_frame']}, {len(entry['heldout_frames'])} held-out frames; "
          f"{entry['scale'] * 1000:.1f} mm per generated unit; alignment RMSE {entry['align_rmse_mm']:.2f} mm "
          f"(limit {ALIGN_LIMIT_SHARE * entry['measured_extent_mm']:.1f} mm)")
    ties = entry["alignment_ties"]
    if len(ties) > 1:
        chosen = next(p for p in ties if p["chosen"])
        others = ", ".join(f"{p['photo_disagreement']:.3f}" for p in ties if not p["chosen"])
        print(f"  {len(ties)} poses fit the shape within {100 * (TIE_SHARE - 1):.0f}% of the best RMSE "
              f"({min(p['rmse_mm'] for p in ties):.3f} to {max(p['rmse_mm'] for p in ties):.3f} mm); the condition photo "
              f"chose the one at {chosen['rmse_mm']:.3f} mm, disagreement {chosen['photo_disagreement']:.3f} against {others}"
              + ("" if chosen["lowest_rmse"] else ", not the pose with the lowest RMSE"))
    else:
        print(f"  one pose fits the shape within {100 * (TIE_SHARE - 1):.0f}% of the best RMSE, so the photo was not needed")
    print(f"  {'':36s}{'generated':>18s}{'measured':>18s}")
    print(f"  {'Chamfer to the scan (mm)':36s}{entry['chamfer_mm']['generated']:18.2f}{entry['chamfer_mm']['measured']:18.2f}")
    for label, g, m in (("held-out base color, PSNR / LPIPS", held["generated"], held["measured"]),
                        ("held-out gain-matched, PSNR / LPIPS", held["generated_matched"], held["measured_matched"])):
        print(f"  {label:36s}{g['psnr']:10.2f} / {g['lpips']:.3f}{m['psnr']:10.2f} / {m['lpips']:.3f}")
    relit = held["measured_relit"]
    print(f"  {'held-out relit, from stage 4b':36s}{'':18s}{relit['psnr']:10.2f} / {relit['lpips']:.3f}")
    for note in generated_notes(entry):
        print("  " + note)
(WORK / "generated_vs_measured.json").write_text(json.dumps(comparisons, indent=2) + "\n")
print(f"compared {len(comparisons)} generated objects in {time.time() - t:.0f} s")

In [ ]:
import io

import matplotlib.pyplot as plt
from IPython.display import Image as IPImage, display

# for each object, the held-out frame where the generated mesh loses most to the measured one
fig, axes = plt.subplots(len(comparisons), 3, figsize=(9.75, 3.4 * len(comparisons)), squeeze=False)
for row, entry in zip(axes, comparisons):
    per = {f["name"]: f for f in entry["held_out_per_frame"]}
    name = min(per, key=lambda n: per[n]["generated_matched"]["psnr"] - per[n]["measured_matched"]["psnr"])
    r = generated_extras[entry["object"]]["renders"][name]
    ys, xs = np.nonzero(r["mask"])
    box = (slice(max(ys.min() - 20, 0), ys.max() + 20), slice(max(xs.min() - 20, 0), xs.max() + 20))
    for ax, key in zip(row, ("photo", "generated", "measured")):
        ax.imshow((r[key] if key == "photo" else np.where(r["mask"][..., None], r[key], 1.0))[box])
        score = per[name].get(f"{key}_matched")
        ax.set_title(f"{entry['object']} {name}" if score is None else
                     f"{key}, gain-matched\n{score['psnr']:.1f} dB, LPIPS {score['lpips']:.3f}", fontsize=9)
        ax.axis("off")
fig.tight_layout()
buf = io.BytesIO()
fig.savefig(buf, format="jpg", dpi=80)
plt.close(fig)
display(IPImage(data=buf.getvalue()))

In [ ]:
import matplotlib


def log_generated(index: int, obj: str, extras: dict, max_mm: float = 10.0):
    """Under generated/<obj>, side by side: the aligned generated mesh with its texture, the same mesh
    and the measured stage 4 mesh colored by distance to the placed scan (0 to max_mm), and the scan."""
    root = f"generated/{obj}"
    placed, measured, scan = extras["placed"], extras["measured_uv"], extras["scan_points"]
    tree = cKDTree(scan)
    colormap = matplotlib.colormaps["viridis"]
    colors = lambda v: (colormap(np.clip(tree.query(v)[0] * 1000 / max_mm, 0, 1))[:, :3] * 255).astype(np.uint8)
    texture = (cv2.resize(extras["generated_color"], (1024, 1024), interpolation=cv2.INTER_AREA) * 255).astype(np.uint8)
    mesh = lambda m, **kw: rr.Mesh3D(vertex_positions=m["vertices"], triangle_indices=m["faces"],
                                     vertex_normals=m["normals"], **kw)
    slots = (("generated_texture", mesh(placed, vertex_texcoords=placed["uvs"], albedo_texture=texture)),
             ("generated_distance", mesh(placed, vertex_colors=colors(placed["vertices"]))),
             ("measured_distance", mesh(measured, vertex_colors=colors(measured["vertices"]))),
             ("scan", rr.Points3D(scan[:: max(1, len(scan) // 50_000)], colors=[[170, 170, 170]], radii=0.0005)))
    step = 1.4 * np.ptp(measured["vertices"], axis=0).max()
    rr.log(root, rr.Transform3D(translation=[0.0, 0.6 * index, 0.0]), static=True)
    for k, (name, archetype) in enumerate(slots):
        rr.log(f"{root}/{name}", rr.Transform3D(translation=[k * step, 0.0, 0.0]), static=True)
        rr.log(f"{root}/{name}/geometry", archetype, static=True)


rr.log("generated", rr.ViewCoordinates.RIGHT_HAND_Z_UP, static=True)
for i, entry in enumerate(comparisons):
    log_generated(i, entry["object"], generated_extras[entry["object"]])

## Scorecards

The two scorecards collect the numbers that describe the twin. The appearance scorecard holds the held-out PSNR and LPIPS of the splat, of the mesh and of the gain-matched mesh, the splat's training-view scores, the second-light comparison, and the metallic decision. Its `refined` block holds the scores of the refined maps and their second-light comparison; when there are no usable refined maps it is null, and `refined_skipped` gives the reason. The behavior scorecard holds the mass you weighed next to the mass the asset and the simulator carry, your ruler dimensions next to the mesh's oriented box (measured as in stage 5, leaving out the outermost 0.1% of the surface on each side), the source of the stage 1 scale, the real and simulated slide angles, the drop result, and which stage 5 mass model was exported.

The real slide angle is the mean of your tilt readings. The simulated one is quantized to the 1 degree ramp step, so a difference of one degree is within the resolution of the test.


In [ ]:
import numpy as np

APPEARANCE_FIELDS = ("heldout_frames", "splat", "mesh", "mesh_gain_matched", "splat_training", "second_light",
                     "metallic", "renderer", "lpips_net")
BEHAVIOR_FIELDS = ("mass_kg", "dims_mm", "scale_source", "slide_angle_deg", "drop", "mu_static", "mass_model",
                   "solver", "assumed")


def robust_extents(mesh, trim: float = 0.001, n: int = 400_000, seed: int = 0) -> np.ndarray:
    """Extents (m) of the mesh along its oriented box axes, leaving out the outermost `trim` share of
    the surface area on each side, as stage 5 measured them."""
    import trimesh

    to_obb, _ = trimesh.bounds.oriented_bounds(mesh)
    points, _ = trimesh.sample.sample_surface(mesh, n, seed=seed)
    q = points @ to_obb[:3, :3].T + to_obb[:3, 3]
    return np.percentile(q, 100 * (1 - trim), axis=0) - np.percentile(q, 100 * trim, axis=0)


def missing_fields(card: dict, fields) -> list[str]:
    """Names of the required fields that are absent or None."""
    return [f for f in fields if card.get(f) is None]

def appearance_card(rows: list[dict], refined: dict | None, refined_skipped: str | None, *, heldout_excluded=(),
                    second_light: dict | None = None, second_light_refined: dict | None = None,
                    second_light_reason: str = "no second-light photo", metallic: dict | None = None) -> dict:
    """The appearance scorecard from the per-frame score rows. The bake rows are held-out means; with
    refined maps (refined_inputs) the "refined" block holds their unlit and gain-matched held-out
    scores, the relit score copied from stage 4b's refine.json, and their second-light comparison.
    Without them "refined" is None and refined_skipped says why."""
    held = [r for r in rows if r["split"] == "held-out"]
    trained = [r for r in rows if r["split"] == "training"]
    avg = lambda rs, key: float(np.mean([r[key] for r in rs]))
    score = lambda rs, key: {"psnr_db": avg(rs, f"{key}_psnr"), "lpips": avg(rs, f"{key}_lpips")}
    skipped = {"status": "skipped", "reason": second_light_reason}
    card = {
        "heldout_frames": [r["name"] for r in held],
        "heldout_excluded": [{"name": n, "reason": why} for n, why in heldout_excluded],
        "splat": score(held, "splat"),
        "mesh": score(held, "mesh"),
        "mesh_gain_matched": score(held, "mesh_matched") | {"gain_median": float(np.median([r["gain"] for r in held]))},
        "splat_training": {"frames": [r["name"] for r in trained]} | score(trained, "splat"),
        "second_light": second_light if second_light is not None else skipped,
        "metallic": metallic if metallic is not None else {"source": "not given"},
        "renderer": "nvdiffrast 0.4.0 (NVIDIA Source Code License, non-commercial)",
        "lpips_net": "alex",
    }
    if refined is None:
        if not refined_skipped:
            raise ValueError("without refined maps, refined_skipped must say why they were left out")
        return card | {"refined": None, "refined_skipped": refined_skipped}
    relit = refined["record"]["runs"]["refined"]["relit"]
    matched4b = refined["record"]["runs"]["refined"]["base_color_matched"]
    card["refined"] = {
        "mesh": score(held, "refined"),
        "mesh_matched": score(held, "refined_matched")
        | {"gain_median": float(np.median([r["refined_gain"] for r in held]))},
        "relit": {"psnr_db": relit["psnr"], "lpips": relit["lpips"],
                  "stage4b_base_color_matched": {"psnr_db": matched4b["psnr"], "lpips": matched4b["lpips"]},
                  "source": "from stage 4b (stage4b/refine.json): rendered there under the light it fitted to the "
                            "training frames, not measured again here; compare it with stage4b_base_color_matched, "
                            "which stage 4b scored with the same renderer"},
        "second_light": second_light_refined if second_light_refined is not None else skipped,
        "mesh_uv_sha256": refined["record"].get("mesh_uv_sha256"),
    }
    return card

In [ ]:
import trimesh

metallic = {"source": "baked" if metallic_source == "baked" else "set to 0", "baked_mean": float(baked_metallic.mean()),
            "reason": "physical.json names materials" if metallic_source == "baked"
            else "physical.json names no materials, so the object is treated as non-metal"}
appearance = appearance_card(rows, refined, refined_skipped, heldout_excluded=heldout_excluded,
                             second_light=second_light, second_light_refined=second_light_refined,
                             second_light_reason=second_light_reason, metallic=metallic)
mesh3 = trimesh.load(STAGE3 / "mesh.ply", process=False)
measured_dims = sorted(float(x) for x in physical["dims_mm"].values())
mesh_dims = sorted(float(x) for x in robust_extents(mesh3) * 1e3)
behavior_card = {
    "mass_kg": {"measured": float(physical["mass_kg"]), "asset": float(asset["mass_kg"]),
                "simulator": drop["run"]["mass_kg"]},
    "dims_mm": {"measured": measured_dims, "mesh": mesh_dims,
                "off_pct": [100 * (m / r - 1) for m, r in zip(mesh_dims, measured_dims)]},
    "scale_source": scale["source"],
    "slide_angle_deg": {"real": float(tilt.mean()), "real_readings": tilt.tolist(), "real_spread": float(tilt.std(ddof=1)),
                        "atan_mu": float(np.degrees(np.arctan(physics["mu_static"]))), "sim": sim_angle,
                        "sim_step": 1.0, "sim_criterion": "more than 5 mm along the ramp in 1 s"},
    "drop": {k: v for k, v in drop.items() if k != "run"},
    "mu_static": float(physics["mu_static"]),
    "mass_model": physics["model"],
    "solver": "Newton 1.7.0, XPBD, 10 iterations, dt 1/600 s",
    "assumed": bool(physical.get("assumed", False)),
}
print("appearance (held-out, masked):")
for key in ("splat", "mesh", "mesh_gain_matched"):
    print(f"  {key:18s} PSNR {appearance[key]['psnr_db']:6.2f} dB, LPIPS {appearance[key]['lpips']:.3f}")
if appearance["refined"] is None:
    print(f"  refined maps: not scored ({appearance['refined_skipped']})")
else:
    for key, label in (("mesh", "refined"), ("mesh_matched", "refined_matched"), ("relit", "refined_relit")):
        print(f"  {label:18s} PSNR {appearance['refined'][key]['psnr_db']:6.2f} dB, "
              f"LPIPS {appearance['refined'][key]['lpips']:.3f}" + (" (from stage 4b)" if key == "relit" else ""))
print(f"  second light: {appearance['second_light'].get('reason') or 'compared'}")
print(f"  metallic: {appearance['metallic']['source']} ({appearance['metallic']['reason']})")
print("behavior:")
print(f"  mass: measured {behavior_card['mass_kg']['measured']} kg, asset {behavior_card['mass_kg']['asset']:.4f} kg, "
      f"simulator {behavior_card['mass_kg']['simulator']:.4f} kg ({physics['model'].replace('_', '-')} mass model)")
print(f"  dimensions: measured {measured_dims} mm, mesh {np.round(mesh_dims, 1).tolist()} mm "
      f"({', '.join(f'{x:+.1f}%' for x in behavior_card['dims_mm']['off_pct'])}); scale from the {scale['source']}")
print(f"  slide angle: real {tilt.mean():.1f} degrees, simulated {sim_angle:.0f} degrees (1 degree steps), "
      f"atan(mu) {behavior_card['slide_angle_deg']['atan_mu']:.1f} degrees")
print(f"  drop: {'comes to rest' if drop['rest'] else 'does not come to rest'}")
if behavior_card["assumed"]:
    print("  the measured values for this object are assumed, not measured (physical.json)")

## The asset in the viewer

The viewer shows the exported mesh with its base color under `world/asset`, in the same world frame as every earlier stage. Under `sim` it replays three simulations along the `sim_time` timeline, drawn with the convex pieces the simulator collides: the drop onto the ground, the ramp at the last angle that holds, and the ramp at the first angle that slides. The `heldout` timeline steps through the held-out frames with the photo, the splat render and the mesh render side by side, plus the render of the refined base color when stage 4b ran. The generated and measured meshes of the reference objects are under `generated`, as described in the previous section.

In [ ]:
def log_stage6(mesh_uv, base_color, pieces, com_m, drop, hold, slide, renders):
    texture = (cv2.resize(base_color, (1024, 1024), interpolation=cv2.INTER_AREA) * 255).astype(np.uint8)
    rr.log("world/asset", rr.Mesh3D(vertex_positions=mesh_uv["vertices"], triangle_indices=mesh_uv["faces"],
                                    vertex_normals=mesh_uv["normals"], vertex_texcoords=mesh_uv["uvs"],
                                    albedo_texture=texture), static=True)
    rr.log("sim", rr.ViewCoordinates.RIGHT_HAND_Z_UP, static=True)
    palette = np.array([[31, 119, 180], [255, 127, 14], [44, 160, 44], [214, 39, 40], [148, 103, 189],
                        [140, 86, 75], [227, 119, 194], [127, 127, 127], [188, 189, 34], [23, 190, 207]])
    trials = {"drop": (drop, np.eye(3), np.zeros(3))}
    if slide is not None:
        trials["hold"], trials["slide"] = (hold, hold["ramp_rotation"], np.array([0.0, 0.4, 0.0])), \
                                          (slide, slide["ramp_rotation"], np.array([0.0, 0.8, 0.0]))
    for name, (trial, Rr, offset) in trials.items():
        run = trial["run"]
        if name != "drop":
            rr.log(f"sim/{name}/ramp", rr.Boxes3D(half_sizes=[[0.25, 0.12, 0.005]], centers=[offset + Rr @ [0.1, 0, -0.005]],
                                                  rotation_axis_angles=[rr.RotationAxisAngle(axis=[0, 1, 0],
                                                                        degrees=float(np.degrees(np.arctan2(Rr[0, 2], Rr[0, 0]))))],
                                                  colors=[[200, 200, 200]], fill_mode="solid"), static=True)
        else:
            rr.log("sim/drop/ground", rr.Boxes3D(half_sizes=[[0.15, 0.15, 0.002]], centers=[[0, 0, -0.002]],
                                                 colors=[[200, 200, 200]], fill_mode="solid"), static=True)
        for i, (v, f) in enumerate(pieces):  # the colliders, placed relative to the body's center of mass
            rr.log(f"sim/{name}/object/piece_{i:02d}", rr.Mesh3D(vertex_positions=v - com_m, triangle_indices=f,
                                                                 albedo_factor=palette[i % len(palette)]), static=True)
        for k in range(0, len(run["com"]), 10):  # every 10th step: 60 frames per simulated second
            rr.set_time("sim_time", duration=k * run["dt"])
            rr.log(f"sim/{name}/object", rr.Transform3D(translation=offset + run["com"][k], mat3x3=run["rotation"][k]))
    for i, (name, r) in enumerate(renders.items()):
        rr.set_time("heldout", sequence=i)
        k = 480 / r["photo"].shape[1]
        for key in ("photo", "splat", "mesh", "refined"):
            if key not in r:  # no refined maps in this project
                continue
            img = cv2.resize((r[key] * 255).astype(np.uint8), None, fx=k, fy=k, interpolation=cv2.INTER_AREA)
            rr.log(f"heldout/{key}", rr.EncodedImage(contents=cv2.imencode(".jpg", img[..., ::-1])[1].tobytes(),
                                                     media_type="image/jpeg"))


log_stage6(mesh_uv, base_color, pieces, np.asarray(physics["com_m"], dtype=float), drop, hold, slide, renders)

In [ ]:
show_rerun()

## One recording for the whole pipeline

Each stage saved its own log under one recording id, so the stage logs describe one recording in pieces. Merging them gives a single file that replays the whole pipeline, from the frames and cameras of stage 1 to the simulations here, in one viewer.

The merged file must stay under 500 MB so that it opens in a browser, and the asset folder under 200 MB. The cell lists the size of each stage's log, merges them, and measures the merged file, which can be smaller than the sum because data that every stage logs once is stored once. If the merged file is over the limit it is deleted, and the cell names the stage that logged the most and the setting in that stage's logging cell that controls its size: an image width, the number of splat centers, or a texture size. Lower it, rerun that stage's logging cell, and merge again.

The merge contains the logs that exist in your project. When you run every stage on your own object, those are your seven runs, stage 4b included. On the reference object shown on this page, the logs of stages 1 to 5 and 4b come from separate earlier runs of those stages on the same reference data, so the merged recording joins seven runs rather than one. A project that falls back to the reference holds only the logs of the stages it ran itself, and its merge lists the others as missing.


In [ ]:
import subprocess
import sys
from pathlib import Path

RRD_LIMIT_MB, USD_LIMIT_MB = 500, 200
STAGE_KEYS = ("1", "2", "3", "4", "4b", "5", "6")  # pipeline order; the log of stage <key> is rrd/stage<key>.rrd
LOG_SETTINGS = {
    "1": "stage 1: the camera thumbnail width in log_stage1 (320 px), or log fewer cameras",
    "2": "stage 2: the frame width in log_stage2 (480 px), or log every second frame",
    "3": "stage 3: the splat centers per splat in splat_points (limit, 50,000), or decimate the mesh",
    "4": "stage 4: the texture size in log_stage4 (1024 px), or log fewer bake views",
    "4b": "stage 4b: the held-out image width in log_stage4b (480 px), or log the refined texture smaller",
    "5": "stage 5: log the decimated mesh and fewer convex pieces in log_stage5",
    "6": "stage 6: the held-out image width in log_stage6 (480 px), or log fewer simulation frames",
}


def stage_log_sizes(rrd_dir: Path) -> dict:
    """Bytes of each stage's log rrd/stage<key>.rrd that exists, keyed by the stage key ("1" to "6", "4b")."""
    return {k: (Path(rrd_dir) / f"stage{k}.rrd").stat().st_size for k in STAGE_KEYS
            if (Path(rrd_dir) / f"stage{k}.rrd").is_file()}


def folder_bytes(folder: Path) -> int:
    return sum(f.stat().st_size for f in Path(folder).rglob("*") if f.is_file())


def size_problems(sizes: dict, merged_bytes: int | None, usd_bytes: int,
                  rrd_limit_mb: float = RRD_LIMIT_MB, usd_limit_mb: float = USD_LIMIT_MB) -> list[str]:
    """Refusals for the hand-in: the merged recording over its limit (naming the stage that logged the
    most and the setting to change), and the USD folder over its limit. merged_bytes is the merged
    file's size; with None the sum of the logs stands in for it (merging removes duplicate static
    data, so the sum is an upper bound)."""
    problems = []
    total = merged_bytes if merged_bytes is not None else sum(sizes.values())
    if total > rrd_limit_mb * 1e6:
        largest = max(sizes, key=sizes.get)
        problems.append(f"the merged recording is {total / 1e6:.0f} MB, over the {rrd_limit_mb:.0f} MB limit; stage {largest} "
                        f"logged the most ({sizes[largest] / 1e6:.0f} MB): reduce {LOG_SETTINGS[largest]} and rerun it")
    if usd_bytes > usd_limit_mb * 1e6:
        problems.append(f"the USD asset folder is {usd_bytes / 1e6:.0f} MB, over the {usd_limit_mb:.0f} MB limit: "
                        "keep the textures at 2048 px or below and decimate the visual mesh")
    return problems


def merge_logs(rrd_dir: Path, out: Path) -> int:
    """Merge every stage log into one recording with `rerun rrd merge`; returns its size in bytes."""
    present = stage_log_sizes(rrd_dir)
    logs = [str(Path(rrd_dir) / f"stage{k}.rrd") for k in STAGE_KEYS if k in present]
    result = subprocess.run([sys.executable, "-m", "rerun", "rrd", "merge", *logs, "-o", str(out)],
                            capture_output=True, text=True)
    if result.returncode != 0:
        raise RuntimeError(f"rerun rrd merge failed: {result.stderr.strip()}")
    return Path(out).stat().st_size

In [ ]:
sizes = stage_log_sizes(PROJECT / "rrd")
for n in STAGE_KEYS:
    print(f"rrd/stage{n}.rrd: " + (f"{sizes[n] / 1e6:.1f} MB" if n in sizes else "missing (that stage was not run in this project)"))
usd_bytes = folder_bytes(WORK / "asset")
merged_bytes = merge_logs(PROJECT / "rrd", WORK / "pipeline.rrd")
problems = size_problems(sizes, merged_bytes, usd_bytes)  # the merged file, not the sum of the logs
if problems:
    (WORK / "pipeline.rrd").unlink()  # refused: the over-limit file is not kept
print(f"merged {len(sizes)} stage logs: {merged_bytes / 1e6:.1f} MB" + ("" if not problems else ", refused")
      + f" (limit {RRD_LIMIT_MB} MB); USD asset folder {usd_bytes / 1e6:.1f} MB (limit {USD_LIMIT_MB} MB)")
for p in problems:
    print("refused:", p)

## Check

The check opens the asset again and requires no validator errors, one meter per unit with Z up, a rigid body, and the stage 5 mass, center of mass and inertia tensor rebuilt from the principal moments and axes. Every convex piece must be a collider with the convex hull approximation, and all of them must carry your friction coefficient. In the simulator the body must load with the measured mass and come to rest after the drop. Both scorecards must have every field, the appearance scorecard must hold the refined scores or the reason they are missing, and the size limits must hold. Each generated mesh must lie on its measured mesh, with an alignment RMSE of at most 5% of the measured mesh's longest side. The slide angles and the appearance scores have no pass threshold: they are measurements of the twin, and the questions ask you to explain them. Only when everything passes is `stage6/` written, with the asset, the two scorecards, the generated-against-measured comparison and the merged recording.

In [ ]:
import numpy as np


def stage6_problems(usd_errors: list, asset: dict, physics: dict, sim_mass_kg: float, drop: dict,
                    appearance: dict, behavior_card: dict, size_refusals: list) -> list[str]:
    """Failures of the stage 6 check: the asset must open and pass the validators, carry the stage 5
    mass properties, colliders and friction in meters with Z up, load in the simulator with its mass,
    come to rest after a drop, and both scorecards must have every field. Missing refined maps are not
    a failure, but the appearance card must then say why."""
    failures = [f"OpenUSD validation (errors and warnings both fail): {e}" for e in usd_errors]
    if asset["meters_per_unit"] != 1.0 or asset["up_axis"] != "Z":
        failures.append(f"the stage is {asset['meters_per_unit']} m per unit with up axis {asset['up_axis']}; "
                        "it must be 1 m per unit, Z up")
    if not asset["rigid_body"]:
        failures.append("the default prim has no RigidBodyAPI")
    if not np.isclose(asset["mass_kg"], physics["mass_kg"], rtol=1e-5):
        failures.append(f"the asset's mass is {asset['mass_kg']} kg, stage 5 measured {physics['mass_kg']} kg")
    if np.linalg.norm(asset["com_m"] - np.asarray(physics["com_m"])) > 1e-6:
        failures.append("the asset's center of mass differs from stage 5's")
    I = np.asarray(physics["inertia_kgm2"], dtype=float)
    if np.linalg.norm(asset["inertia_kgm2"] - I) > 1e-4 * np.linalg.norm(I):
        failures.append("the inertia rebuilt from diagonalInertia and principalAxes is not stage 5's tensor: "
                        "check the eigenvector order and the quaternion")
    if not asset["colliders"] or any(c["approximation"] != "convexHull" for c in asset["colliders"]):
        failures.append("every convex piece must be a collider with the convexHull approximation")
    mu = physics["mu_static"]
    if len(asset["friction"]) != 1 or not np.allclose(asset["friction"][0], (mu, mu), rtol=1e-5):
        failures.append(f"the colliders' friction is {asset['friction']}, expected static and dynamic {mu:.3f}")
    if not np.isclose(sim_mass_kg, physics["mass_kg"], rtol=1e-4):
        failures.append(f"the simulator loaded a body of {sim_mass_kg} kg, not {physics['mass_kg']} kg")
    if not drop["rest"]:
        failures.append(f"the dropped object does not come to rest (speed {drop['final_speed_m_s'] * 1e3:.1f} mm/s, "
                        f"spin {drop['final_spin_rad_s']:.2f} rad/s): check the colliders and the inertia")
    for name, card, fields in (("appearance", appearance, APPEARANCE_FIELDS),
                               ("behavior", behavior_card, BEHAVIOR_FIELDS)):
        missing = missing_fields(card, fields)
        if missing:
            failures.append(f"the {name} scorecard is missing {', '.join(missing)}")
    metallic = appearance.get("metallic")
    if isinstance(metallic, dict) and metallic.get("source") == "not given":
        failures.append("the appearance scorecard does not record the metallic decision")
    if appearance.get("refined") is None and not appearance.get("refined_skipped"):
        failures.append("the appearance scorecard has no refined block and no refined_skipped reason")
    return failures + list(size_refusals)


def write_stage6(out, work, appearance: dict, behavior_card: dict) -> None:
    """Move the asset, the merged recording and the generated-against-measured comparison from the
    work folder into stage6/ in one step, with the two scorecards."""
    import json
    import shutil
    from pathlib import Path

    with atomic_dir(out) as d:
        shutil.copytree(Path(work) / "asset", d / "asset")
        if (Path(work) / "pipeline.rrd").exists():
            shutil.copyfile(Path(work) / "pipeline.rrd", d / "pipeline.rrd")
        if (Path(work) / "generated_vs_measured.json").exists():
            shutil.copyfile(Path(work) / "generated_vs_measured.json", d / "generated_vs_measured.json")
        to_json = lambda card: json.dumps(card, indent=2, default=lambda x: x.tolist() if hasattr(x, "tolist") else str(x))
        (d / "scorecard_appearance.json").write_text(to_json(appearance) + "\n")
        (d / "scorecard_behavior.json").write_text(to_json(behavior_card) + "\n")

In [ ]:
failures = stage6_problems(usd_errors, read_usd(USD), physics, drop["run"]["mass_kg"], drop,
                           appearance, behavior_card, problems)
failures += [p for entry in comparisons for p in generated_problems(entry)]
print(f"USD: opens in usd-core, {len(usd_errors)} validator errors or warnings; {asset['meters_per_unit']} m per unit, "
      f"up axis {asset['up_axis']}; {len(asset['colliders'])} convexHull colliders; mass {asset['mass_kg']:.4f} kg")
print(f"Newton: loaded one body of {drop['run']['mass_kg']:.4f} kg; drop "
      f"{'comes to rest' if drop['rest'] else 'does not come to rest'}; slides at {sim_angle:.0f} degrees "
      f"(real {tilt.mean():.1f}, no threshold)")
print(f"scorecards: appearance {len(APPEARANCE_FIELDS)} fields, behavior {len(BEHAVIOR_FIELDS)} fields; missing: "
      f"{missing_fields(appearance, APPEARANCE_FIELDS) + missing_fields(behavior_card, BEHAVIOR_FIELDS) or 'none'}")
print("refined maps:", "scored beside the bake" if appearance["refined"] else f"not scored ({appearance['refined_skipped']})")
print("generated against measured: " + "; ".join(
    f"{e['object']} alignment RMSE {e['align_rmse_mm']:.2f} mm (limit {ALIGN_LIMIT_SHARE * e['measured_extent_mm']:.1f})"
    for e in comparisons))
print(f"sizes: merged recording {merged_bytes / 1e6 if merged_bytes else float('nan'):.1f} MB, USD folder "
      f"{usd_bytes / 1e6:.1f} MB")
print(f"stage time {(time.time() - T0) / 60:.1f} min")
assert not failures, "; ".join(failures)  # nothing is written when a gate fails
write_stage6(PROJECT / "stage6", WORK, appearance, behavior_card)
shutil.rmtree(WORK, ignore_errors=True)
if REFERENCE:
    (PROJECT / "stage6" / ".reference").touch()
print("passed")

### Question 1. Held-out LPIPS

Explain why the appearance of a reconstruction is scored on held-out views and not on the views it was trained on. What can a Gaussian splat do during training that raises its training-view score without making it look better from a new viewpoint? Then compare PSNR and LPIPS: what each one measures, which kinds of error (a small misalignment, blur, a wrong color, floaters near the object) each punishes more, and why the mask matters for both. Point to your splat's training and held-out scores and to the mesh's scores, and say what the gap between them tells you.

*Write your answer here.*

### Question 2. Real and simulated slide angles

The simulator and your ramp both obey $\tan\theta = \mu$ in theory, yet the two angles can differ. List the places where the simulated test departs from the real one: how a position-based solver applies friction and why it creeps below the slip angle, how the contact patch of the convex pieces differs from the real base of the object, the equal static and dynamic friction in the asset, the 1 degree step and the 5 mm criterion, and the spread of your own readings. Say which of these can move the simulated angle up and which down. Point to the real and simulated angles and the creep your cells printed.

*Write your answer here.*

### Question 3. Where the twin falls short

Each scorecard number tests only part of the pipeline. For each of the following, say what it can detect and what it cannot, and name the stage whose assumption limits it: the gain-matched mesh score (consider an albedo that is wrong by one overall factor), the second-light comparison (consider a shiny object), the dimensions against your ruler, the drop test, and the slide angle (consider an object whose center of mass is in the wrong place). Then write the shortfalls section in your `submission.md`, as the project page describes: the two or three places where your twin differs most from your object, with the number that shows it and the stage it comes from. The answer cell below holds the first part; the shortfalls section goes only in `submission.md`.

*Write your answer here.*

### Question 4. Generated and measured

A single-image generator sees one photo of an object. Explain what it can and cannot know from that photo about the object's geometry, its size and its materials, and why. For size, use the pinhole projection: show which change to the object and its distance leaves the photo unchanged, say what in this project's capture and pipeline removes that ambiguity, and explain why the generated mesh's scale therefore had to come from stage 1 and could not come from the generated mesh or the condition photo. Then compare the two kinds of score: what the Chamfer distance to the scan rewards that the held-out photo scores do not, what the photo scores reward that Chamfer does not, and how one mesh can win on one and lose on the other. Point to the row of `generated_vs_measured.json` where the generated object wins or loses most against the measured one, and to the part of the object, seen in the viewer, that explains it.

*Write your answer here.*